## AI Chat App: কীভাবে বানালাম

**Phase 02 — Project Notebook**

এই notebook-এ আমার `chat_app/app` project-টা শুরু থেকে শেষ পর্যন্ত বোঝানো আছে।
User signup আর login করে, নিজের conversation বানায়, message পাঠায়, আর LLM উত্তর দেয়। সব কিছু PostgreSQL-এ জমা থাকে।

প্রতিটা অংশে একই নিয়ম: **গল্প** → **ধারণা** → **এক লাইনে**।
প্রতিটা file আর endpoint-এ চারটা জিনিস: **(ক) আসল code**, **(খ) কোন লাইন কেন**, **(গ) কেন এই উপায়, বিকল্প কী**, **(ঘ) Swagger `/docs`-এ কীভাবে test করবো**।

এই notebook শুধু পড়ার জন্য, কোনো cell run করতে হবে না। সব code `chat_app/app` folder-এর file থেকে হুবহু তোলা।

### সূচিপত্র (Table of Contents)

**[১. বড় ছবি](#p1)**
- [App-টা কী করে](#s1)
- [Restaurant-এর গল্প: কে কোন কাজ করে](#s2)
- [একটা request-এর জীবন (signup, login, protected route, message)](#s3)

**[২. Folder ও file-এর map](#p2)**
- [কোন file-এর কী একটাই কাজ](#s4)

**[৩. File ধরে ধরে (যে ক্রমে বানিয়েছি)](#p3)**
- [Docker: chat-db (PostgreSQL, port 5434)](#s5)
- [models.py: গুদামের তাক](#s6)
- [create_table.py: তাক বসানো](#s7)
- [.env ও .gitignore: গোপন জিনিস Git-এর বাইরে](#s8)
- [database.py: গুদামের চাবি](#s9)
- [security.py: locker আর coupon](#s10)
- [schemas.py: order form আর receipt](#s11)
- [llm.py: বাইরের রাঁধুনিকে ডাকা](#s12)
- [main.py: শুরুর অংশ (import, app, bearer_scheme)](#s13)
- [GET /](#s14)
- [POST /signup](#s15)
- [POST /login](#s16)
- [get_current_user: দরজার guard](#s17)
- [GET /me](#s18)
- [POST /conversations](#s19)
- [GET /conversations](#s20)
- [POST /conversations/{id}/messages](#s21)
- [GET /conversations/{id}/messages](#s22)

**[৪. Security notes](#p4)**
- [চারটা নিয়ম যা ভাঙা যাবে না](#s23)

**[৫. বানাতে গিয়ে যে ভুলগুলো করেছি](#p5)**
- [`=` বনাম `:` (existing:)](#m1)
- [`db.commit`-এ () নেই](#m2)
- [অন্য file থেকে import ভুলে যাওয়া](#m3)
- [code-এ `...` রয়ে গেছে (Ellipsis)](#m4)
- [`in None` বনাম `is None`](#m5)
- [`.filter()` আছে, `.first()` নেই](#m6)
- [`return`-এর পরে লাইন](#m7)
- [Variable দিয়ে class ঢেকে দেওয়া (Token = ...)](#m8)
- [env variable-এর নাম মেলেনি (SECRET বনাম SECRET_KEY)](#m9)
- [`credentials` বনাম `credentials.credentials`](#m10)
- [token-এর লাইন failure-এর if-এর ভিতরে](#m11)
- [response_model ছাড়া /me থেকে hash বেরিয়ে গেল](#m12)
- [Docker Desktop বন্ধ, তাই chat-db বন্ধ](#m13)

**[৬. 500 error কীভাবে debug করবো](#p6)**
- [Traceback পড়ার নিয়ম](#s24)

**[৭. নিজেকে যাচাই](#p7)**
- [১০টা প্রশ্ন (উত্তর লুকানো)](#s25)

**[৮. বাকি কাজ](#p8)**
- [Checklist](#s26)

<a id="p1"></a>

## ১. বড় ছবি

পুরো app-টা আগে উপর থেকে একবার দেখি। তারপর একটা একটা file খুলবো।

<a id="s1"></a>

### App-টা কী করে

**গল্প:** ধরো একটা restaurant। তুমি গিয়ে নাম লেখাও (signup)। দরজায় পরিচয় দেখালে একটা coupon পাও (login)। এই coupon দেখিয়ে ভিতরে টেবিল নাও (conversation)। তারপর যত খুশি order দাও (message), রান্নাঘর থেকে খাবার আসে (LLM-এর উত্তর)। পরের দিন এসে আগের সব order-এর হিসাব দেখতে পারো (history)।

**ধারণা:** App-টা একটা FastAPI backend। এটা যা যা পারে:

| কাজ | Endpoint | Login লাগে? |
|---|---|---|
| App চলছে কিনা দেখা | `GET /` | না |
| নতুন account | `POST /signup` | না |
| Login করে token নেওয়া | `POST /login` | না |
| আমি কে, দেখা | `GET /me` | হ্যাঁ |
| নতুন conversation | `POST /conversations` | হ্যাঁ |
| আমার সব conversation | `GET /conversations` | হ্যাঁ |
| Message পাঠানো, LLM-এর উত্তর পাওয়া | `POST /conversations/{id}/messages` | হ্যাঁ |
| একটা conversation-এর সব message | `GET /conversations/{id}/messages` | হ্যাঁ |

**যা দিয়ে বানানো:**
- **FastAPI**: endpoint বানানো, Swagger `/docs` নিজে থেকেই আসে।
- **PostgreSQL** (Docker container `chat-db`-এ): data রাখা।
- **SQLAlchemy**: Python class দিয়ে table চালানো।
- **Pydantic**: input আর output-এর আকার ঠিক করা।
- **passlib (bcrypt)**: password hash।
- **python-jose**: JWT token বানানো ও যাচাই।
- **httpx**: Groq API-তে LLM call (model `openai/gpt-oss-20b`)।

> **এক লাইনে:** User login করে নিজের conversation-এ কথা বলে, LLM উত্তর দেয়, আর সব DB-তে জমা থাকে।

<a id="s2"></a>

### Restaurant-এর গল্প: কে কোন কাজ করে

**গল্প:** একটা restaurant-এ সবাই সব কাজ করে না। Waiter রান্না করে না, guard হিসাব রাখে না। প্রত্যেকের একটাই কাজ। আমার code-ও সেভাবে ভাগ করা।

| Restaurant-এ | আমার code-এ | কাজ |
|---|---|---|
| খদ্দের | Client (Swagger, browser) | request পাঠায় |
| **Waiter** | `main.py` | order নেয়, ঠিক লোকের কাছে পাঠায়, উত্তর ফেরত দেয় |
| **Order form আর receipt** | `schemas.py` | খদ্দের কী দিতে পারবে (Create), কী ফেরত পাবে (Response) |
| **Locker** | `security.py` | password-এর ছাপ রাখে, coupon বানায় ও যাচাই করে |
| **গুদাম (warehouse)** | `models.py` + PostgreSQL | কোন তাকে কী থাকবে (table), আর আসল জিনিস |
| **গুদামের চাবি** | `database.py` | প্রতি request-এ চাবি দেয়, কাজ শেষে ফেরত নেয় |
| **Coupon** | JWT token | "এই লোক login করেছে", মেয়াদ 30 মিনিট |
| **দরজার guard** | `get_current_user` | coupon দেখে ঠিক করে ভিতরে ঢুকতে দেবে কিনা |
| বাইরের রাঁধুনি | `llm.py` → Groq | উত্তর রান্না করে |

এই map মনে থাকলে বাকি notebook সহজ লাগবে। প্রতিটা section-এ এই নামগুলোই ব্যবহার করবো।

> **এক লাইনে:** প্রতিটা file-এর একটাই কাজ, ঠিক যেন restaurant-এর একেকজন কর্মী।

<a id="s3"></a>

### একটা request-এর জীবন (signup, login, protected route, message)

**গল্প:** খদ্দের order দেয়, তারপর order-টা কয়েক হাত ঘুরে খাবার হয়ে ফেরত আসে। একটা HTTP request-ও এভাবে কয়েকটা file ঘুরে response হয়ে ফেরে।

**১. Signup** (`POST /signup`)

```
Client ── {"email", "password"} ──▶ main.py signup()            (waiter)
                                     │ UserCreate দিয়ে body যাচাই  (order form)  ভুল হলে 422
                                     │ get_db() থেকে db session     (গুদামের চাবি)
                                     │ email আগে থেকে আছে?          থাকলে 400
                                     │ hash_password()               (locker)
                                     │ User(...) → add → commit → refresh   (গুদাম)
                                     ▼
Client ◀── {"id", "email"} ──────── UserResponse দিয়ে ছাঁকা          (receipt)
```

**২. Login** (`POST /login`)

```
Client ── {"email", "password"} ──▶ main.py login()
                                     │ email দিয়ে user খোঁজা (DB)
                                     │ verify_password()             ভুল হলে 401
                                     │ create_access_token({"sub": email})   (coupon বানানো)
                                     ▼
Client ◀── {"access_token", "token_type": "bearer"} ── Token schema
```

**৩. Protected route** (`GET /me`)

```
Client ── Header: Authorization: Bearer <token> ──▶ HTTPBearer (header নেই → 401)
                                     │
                                     ▼
                         get_current_user()                       (দরজার guard)
                                     │ decode_access_token()      token খারাপ বা মেয়াদ শেষ → 401
                                     │ email দিয়ে user খোঁজা       না পেলে 401
                                     ▼
                         me(current_user) → UserResponse          (hash বাদ)
                                     ▼
Client ◀── {"id", "email"}
```

**৪. Message পাঠানো** (`POST /conversations/{id}/messages`)

```
Client ── token + {"content"} ──▶ get_current_user()  → current_user
                                     │
                                     ▼
                         send_message()
                           ১. conversation এই user-এর?         না হলে 404
                           ২. user-এর message DB-তে save
                           ৩. পুরো history DB থেকে আনা (id অনুযায়ী সাজানো)
                           ৪. history → [{"role", "content"}, ...]
                           ৫. get_reply(messages) → Groq LLM  (llm.py)
                           ৬. assistant-এর উত্তর DB-তে save
                                     ▼
Client ◀── {"id", "role": "assistant", "content"} ── MessageResponse
```

একই message flow mermaid দিয়ে (JupyterLab-এ ছবি হয়ে দেখায়। ছবি না দেখালে উপরের text diagram-ই যথেষ্ট):

```mermaid
sequenceDiagram
    participant C as Client
    participant G as get_current_user
    participant W as send_message
    participant D as PostgreSQL
    participant L as llm.py / Groq
    C->>W: POST /conversations/1/messages + Bearer token
    W->>G: token যাচাই
    G->>D: email দিয়ে user খোঁজা
    D-->>G: User
    G-->>W: current_user
    W->>D: conversation এই user-এর? (না হলে 404)
    W->>D: user message save
    W->>D: পুরো history আনা
    W->>L: get_reply(messages)
    L-->>W: reply text
    W->>D: assistant message save
    W-->>C: MessageResponse
```

> **এক লাইনে:** প্রতিটা request যায় waiter → guard → চাবি → গুদাম/locker/রাঁধুনি, তারপর receipt দিয়ে ছেঁকে ফেরত আসে।

<a id="p2"></a>

## ২. Folder ও file-এর map

<a id="s4"></a>

### কোন file-এর কী একটাই কাজ

**গল্প:** রান্নাঘরে মশলার কৌটায় নাম লেখা থাকে, তাই খুঁজতে হয় না। File-এর নাম আর কাজ মাথায় থাকলে bug খোঁজাও সহজ হয়।

```
Phase_02/
└── chat_app/
    └── app/
        ├── .env              ← গোপন মান। Git-এ যায় না
        ├── .gitignore        ← Git-কে বলে কোন file নেবে না
        ├── models.py         ← table-এর নকশা
        ├── create_table.py   ← একবার চালিয়ে table বানানো
        ├── database.py       ← DB connection আর session
        ├── security.py       ← password hash, JWT
        ├── schemas.py        ← input/output-এর আকার
        ├── llm.py            ← Groq API call
        └── main.py           ← সব endpoint
```

| File | একটাই কাজ | Restaurant-এ |
|---|---|---|
| `models.py` | DB-তে কোন table, কোন column | গুদামের তাক |
| `create_table.py` | `models.py` দেখে table বানায় (একবার) | তাক বসানো |
| `.env` | `DATABASE_URL`, `SECRET_KEY`, `GROQ_API_KEY` রাখে | মালিকের সিন্দুক |
| `.gitignore` | `.env` আর `__pycache__/` Git-এ যেতে দেয় না | "প্রবেশ নিষেধ" বোর্ড |
| `database.py` | engine, `SessionLocal`, `get_db` | গুদামের চাবি |
| `security.py` | hash, verify, token বানানো, token পড়া | locker |
| `schemas.py` | request আর response-এর Pydantic model | order form আর receipt |
| `llm.py` | history পাঠিয়ে LLM-এর উত্তর আনা | বাইরের রাঁধুনি |
| `main.py` | endpoint, guard (`get_current_user`) | waiter আর guard |

**Server চালানো:** `main.py`-তে import গুলো `from database import ...` এরকম, অর্থাৎ পাশের file থেকে। তাই terminal-এ আগে `app` folder-এ ঢুকতে হয়:

```
cd D:\AI-Roadmap\Phase_02\chat_app\app
uvicorn main:app --reload
```

তারপর browser-এ `http://127.0.0.1:8000/docs` খুললে Swagger আসে।

> **এক লাইনে:** একটা file = একটা কাজ। তাই ভুল হলে কোথায় খুঁজবো, আগেই জানি।

<a id="p3"></a>

## ৩. File ধরে ধরে (যে ক্রমে বানিয়েছি)

ক্রম: আগে গুদাম (DB), তারপর তাক (models), তারপর চাবি (database.py), locker (security.py), form (schemas.py), রাঁধুনি (llm.py), শেষে waiter (main.py)।

<a id="s5"></a>

### Docker: chat-db (PostgreSQL, port 5434)

**গল্প:** গুদাম বানাতে জমি কিনে দালান তুলতে হয় না। একটা তৈরি ঘর ভাড়া নিলেই হয়, দরকার না হলে তালা মেরে রাখি। Docker container ঠিক এরকম তৈরি ঘর।

**ধারণা:** PostgreSQL Windows-এ install না করে Docker container-এ চালাই। Container-এর নাম `chat-db`, আর আমার computer থেকে এটাকে পাই **port 5434**-এ।

**(ক) Command** (এরকম command দিয়ে container বানানো। `< >`-এর মানগুলো placeholder। আসল মান শুধু `.env`-এ থাকে):

```
docker run --name chat-db -e POSTGRES_USER=<user> -e POSTGRES_PASSWORD=<password> -e POSTGRES_DB=<db_name> -p 5434:5432 -d postgres
```

রোজকার কাজের command:

```
docker ps                 # কোন container চলছে
docker start chat-db      # বন্ধ container চালু
docker stop chat-db       # বন্ধ করা
docker logs chat-db       # Postgres কী বলছে
```

**(খ) কোন অংশ কেন**

| অংশ | কেন |
|---|---|
| `--name chat-db` | নাম দিলে পরে `docker start chat-db` লিখেই চালু করা যায় |
| `-e POSTGRES_...` | প্রথমবার চালু হওয়ার সময় user, password আর database বানায় |
| `-p 5434:5432` | বাইরের 5434 → container-এর ভিতরের 5432। Postgres ভিতরে সবসময় 5432-এ শোনে। বাইরে 5434 নিয়েছি যাতে অন্য Postgres-এর (5432/5433) সাথে ধাক্কা না লাগে |
| `-d` | background-এ চলে, terminal আটকে থাকে না |
| `postgres` | official PostgreSQL image |

তাই `DATABASE_URL`-এ port হবে **5434**, 5432 না।

**(গ) কেন এই উপায়, বিকল্প কী**
- **Windows-এ সরাসরি Postgres install:** চলে, কিন্তু মুছে নতুন করে শুরু করা ঝামেলা, আর version বদলানো কঠিন।
- **SQLite:** একটা file-ই database, setup লাগে না। শেখার জন্য ভালো। কিন্তু production-এ সাধারণত Postgres চলে, তাই শুরু থেকেই Postgres-এ অভ্যাস করা ভালো।
- **Docker-এর সতর্কতা:** `docker rm chat-db` দিলে container-এর সাথে data-ও যায় (volume না দিলে)। `stop` আর `start` করলে data থাকে।

**(ঘ) Test**
- `docker ps`-এ `chat-db` আর STATUS `Up` দেখাবে।
- Swagger-এ `POST /signup` কাজ করলে বুঝবো DB জীবিত। DB বন্ধ থাকলে 500 আসে ([ভুল ১৩](#m13) দেখো)।

> **এক লাইনে:** `chat-db` হলো Docker-এ ভাড়া করা গুদাম। Computer চালু করে আগে Docker Desktop, তারপর `docker start chat-db`।

<a id="s6"></a>

### models.py: গুদামের তাক

**গল্প:** গুদামে জিনিস রাখার আগে তাক বানাতে হয়, আর প্রতিটা তাকে লেবেল লাগে: এখানে খদ্দেরের নাম, ওখানে টেবিল নম্বর, ওখানে order। `models.py` হলো সেই তাকের নকশা।

**ধারণা:** প্রতিটা Python class = DB-র একটা table। প্রতিটা attribute = একটা column। SQLAlchemy এই class দেখে SQL বানায়।

**(ক) আসল code**

```python
from datetime import datetime

from sqlalchemy import DateTime, ForeignKey, String, Text, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    email: Mapped[str] = mapped_column(String(255), unique=True)
    hashed_password: Mapped[str] = mapped_column(String(255))
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), server_default=func.now()
    )


class Conversation(Base):
    __tablename__ = "conversations"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    title: Mapped[str] = mapped_column(String(200))
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), server_default=func.now()
    )


class Message(Base):
    __tablename__ = "messages"
    id: Mapped[int] = mapped_column(primary_key=True)
    conversation_id: Mapped[int] = mapped_column(ForeignKey("conversations.id"))
    role: Mapped[str] = mapped_column(String(20))
    content: Mapped[str] = mapped_column(Text)
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), server_default=func.now()
    )
```

তিনটা table-এর সম্পর্ক:

```
users (id, email, hashed_password, created_at)
  │ একজন user-এর অনেক conversation
  └──< conversations (id, user_id → users.id, title, created_at)
          │ একটা conversation-এ অনেক message
          └──< messages (id, conversation_id → conversations.id, role, content, created_at)
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `class Base(DeclarativeBase)` | সব model-এর মা-class। `Base.metadata`-তে সব table-এর তালিকা জমা হয়, `create_table.py` এটাই ব্যবহার করে |
| `__tablename__ = "users"` | DB-তে table-এর নাম |
| `Mapped[int]` | Python-কে বলে এই column-এর type int। Editor-ও বুঝতে পারে |
| `mapped_column(primary_key=True)` | প্রতিটা row-এর আলাদা নম্বর। Postgres নিজেই 1, 2, 3 বসায় |
| `String(255), unique=True` | email সর্বোচ্চ 255 অক্ষর, আর একই email দুবার থাকতে পারবে না (DB নিজেই আটকায়) |
| `hashed_password` | নামটাই মনে করিয়ে দেয়: এখানে আসল password কখনো না, শুধু hash |
| `DateTime(timezone=True)` | সময়ের সাথে timezone-ও রাখে, যাতে পরে গোলমাল না হয় |
| `server_default=func.now()` | Python না, **DB নিজে** row বানানোর সময় সময় বসায় |
| `ForeignKey("users.id")` | conversation-এর `user_id` অবশ্যই `users` table-এর কোনো `id` হতে হবে। ভুয়া user_id দিলে DB আটকায় |
| `role: String(20)` | `"user"` বা `"assistant"`। LLM-কে history পাঠাতে এই নামগুলোই লাগে |
| `content: Text` | message লম্বা হতে পারে। `Text`-এর দৈর্ঘ্যের সীমা নেই, `String(n)`-এর আছে |

**(গ) কেন এই উপায়, বিকল্প কী**
- **ORM (SQLAlchemy) বনাম raw SQL:** নিজে `CREATE TABLE ...` আর `INSERT ...` লিখতে পারতাম। ORM-এ Python object দিয়েই কাজ হয়, আর SQL injection-এর ভয় কমে।
- **`relationship()` ব্যবহার করিনি:** এটা দিলে `user.conversations` লিখেই সব conversation পাওয়া যেত। এখন দরকার নেই, তাই রাখিনি। এখন query করে আনি।
- **Message-এ `user_id` নেই কেন:** message → conversation → user, এই শিকল দিয়েই মালিক বোঝা যায়। একই তথ্য দুই জায়গায় রাখলে একসময় দুটো আলাদা হয়ে যেতে পারে।

**(ঘ) Test**
- Swagger-এ সরাসরি `models.py` test করার কিছু নেই। `POST /signup` সফল হলে `users` table ঠিক আছে, `POST /conversations` সফল হলে `conversations` ঠিক আছে।
- চাইলে DB-র ভিতরে দেখা যায়: `docker exec -it chat-db psql -U <user> -d <db_name>` তারপর `\dt` (table-এর তালিকা) আর `\d users` (column-এর তালিকা)।

> **এক লাইনে:** `models.py` = গুদামের তাকের নকশা। একটা class মানে একটা table, একটা attribute মানে একটা column।


<a id="s7"></a>

### create_table.py: তাক বসানো

**গল্প:** নকশা কাগজে থাকলে হবে না, একবার মিস্ত্রি ডেকে তাকগুলো বসাতে হয়। বসানো হয়ে গেলে রোজ রোজ মিস্ত্রি লাগে না।

**ধারণা:** এই file একবার চালালে `models.py`-র সব class দেখে Postgres-এ table বানায়।

**(ক) আসল code**

```python
from models import Base
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

database_url = os.getenv("DATABASE_URL")

engine = create_engine(database_url)

Base.metadata.create_all(engine)
```

চালানো (app folder থেকে, একবার):

```
python create_table.py
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `from models import Base` | `Base.metadata`-তে তিনটা table-এর তালিকা আছে |
| `load_dotenv()` | `.env` file পড়ে মানগুলো environment-এ বসায় |
| `os.getenv("DATABASE_URL")` | DB-র ঠিকানা code-এ না লিখে `.env` থেকে আনা |
| `create_engine(database_url)` | DB-র সাথে যোগাযোগের রাস্তা |
| `Base.metadata.create_all(engine)` | যে table নেই সেটা বানায়। যেটা আছে সেটা ছোঁয় না |

**(গ) কেন এই উপায়, বিকল্প কী**
- **আলাদা file কেন:** table একবারই বানাতে হয়। Server যতবার চালু হবে ততবার চালানোর দরকার নেই।
- **সতর্কতা:** `create_all` পুরনো table **বদলায় না**। পরে কোনো table-এ নতুন column যোগ করলে (যেমন usage tracking-এর জন্য) শুধু এই file আবার চালালে column আসবে না।
- **বিকল্প ১:** `main.py`-র শুরুতে `Base.metadata.create_all(bind=engine)`। সহজ, কিন্তু একই সমস্যা।
- **বিকল্প ২: Alembic** (migration tool)। Table-এর প্রতিটা পরিবর্তন version করে রাখে, পুরনো data না মুছেই column যোগ করা যায়। App বড় হলে এটাই লাগবে।

**(ঘ) Test**
- File চালালে কিছু print হয় না। Error না এলেই সফল।
- Swagger-এ `POST /signup` চালালে `relation "users" does not exist` error না এলে বুঝবো table আছে।

> **এক লাইনে:** `create_table.py` একবার চালাই, তাক বসে যায়। পরে column বদলাতে হলে Alembic লাগবে।

<a id="s8"></a>

### .env ও .gitignore: গোপন জিনিস Git-এর বাইরে

**গল্প:** ঘরের চাবি পাপোশের নিচে রেখে গেলে যে-কেউ পায়। GitHub-এ password বা API key রাখা মানে চাবি পাপোশের নিচে রাখা। Public repo-তে key রাখলে কয়েক মিনিটের মধ্যে bot খুঁজে পায়।

**ধারণা:** গোপন মান code-এ লিখি না, `.env` file-এ রাখি। Code শুধু **নাম** দিয়ে মান চায় (`os.getenv("SECRET_KEY")`)। আর `.gitignore` Git-কে বলে `.env` কখনো নেবে না।

**`.env`-এ তিনটা নাম** (মান এখানে কখনো লিখবো না):

| নাম | কে ব্যবহার করে | কীসের জন্য |
|---|---|---|
| `DATABASE_URL` | `database.py`, `create_table.py` | Postgres-এর ঠিকানা: `postgresql://<user>:<password>@localhost:5434/<db_name>` এই আকারে |
| `SECRET_KEY` | `security.py` | JWT-তে সই করার গোপন চাবি |
| `GROQ_API_KEY` | `llm.py` | Groq API ব্যবহারের অনুমতি |

**(ক) আসল code: `.gitignore`**

```
.env
__pycache__/
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `.env` | তিনটা গোপন মান Git-এ যাবে না |
| `__pycache__/` | Python-এর বানানো `.pyc` file। প্রতিবার বদলায়, Git-এ রাখার মানে নেই |

**(গ) কেন এই উপায়, বিকল্প কী**
- **একবার commit হয়ে গেলে কী হয়:** পরে file মুছলেও Git history-তে থেকে যায়। তখন একমাত্র উপায় key বদলে ফেলা (rotate)।
- **`.env.example`:** একই নাম কিন্তু খালি মান দিয়ে একটা file Git-এ রাখা যায়। অন্য কেউ project নামালে বুঝবে কোন কোন মান লাগবে।
- **Production-এ:** server-এর environment variable বা secret manager ব্যবহার হয়, `.env` file না।

**(ঘ) Test**
- `git status`-এ `.env` কখনো দেখাবে না।
- `git check-ignore -v chat_app/app/.env` (Phase_02 folder থেকে) চালালে কোন `.gitignore`-এর কোন লাইন এটাকে আটকাচ্ছে, সেটা দেখায়।
- Swagger-এ: নাম ভুল হলে server চালু হয় ঠিকই, কিন্তু login-এ 500 আসে। দেখো [ভুল ৯](#m9)।

> **এক লাইনে:** Code-এ শুধু নাম, মান থাকে `.env`-এ, আর `.env` কখনো Git-এ না।

<a id="s9"></a>

### database.py: গুদামের চাবি

**গল্প:** গুদামের একটাই দরজা, কিন্তু অনেক কর্মী। প্রত্যেকে কাজ শুরুর আগে চাবি নেয়, কাজ শেষে ফেরত দেয়। কেউ চাবি নিয়ে বাড়ি চলে গেলে একসময় চাবি ফুরিয়ে যায়, তখন কেউ আর ঢুকতে পারে না।

**ধারণা:** প্রতিটা request-এ একটা DB **session** খুলি, কাজ শেষে বন্ধ করি। `get_db` এই কাজটা নিশ্চিত করে, error হলেও।

**(ক) আসল code**

```python
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))
SessionLocal = sessionmaker(bind=engine)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `load_dotenv()` | `.env` থেকে `DATABASE_URL` পড়া |
| `engine = create_engine(...)` | DB-র সাথে connection-এর একটা pool। পুরো app-এ একটাই engine |
| `SessionLocal = sessionmaker(bind=engine)` | session বানানোর ছাঁচ। `SessionLocal()` ডাকলে নতুন session (নতুন চাবি) |
| `db = SessionLocal()` | এই request-এর জন্য চাবি নেওয়া |
| `yield db` | endpoint-কে চাবি দেওয়া, আর endpoint শেষ না হওয়া পর্যন্ত এখানে থেমে থাকা |
| `finally: db.close()` | endpoint সফল হোক বা error দিক, চাবি ফেরত দেওয়া |

**`yield` আর `finally` কীভাবে একসাথে কাজ করে:**

```
request আসে → get_db() চলে → db = SessionLocal() → yield db  (এখানে থামে)
           → endpoint db ব্যবহার করে
           → endpoint শেষ → get_db() আবার চলে → finally: db.close()
```

`return db` দিলে function সাথে সাথে শেষ হয়ে যেত, পরে `close()` করার কোনো সুযোগ থাকত না। `yield` থেমে থাকে, তাই কাজ শেষে ফিরে এসে বন্ধ করা যায়। এটা Week 08-এর generator-এর মতোই।

একটা মজার তথ্য: একই request-এ `get_current_user` আর endpoint দুজনেই `Depends(get_db)` চায়। FastAPI একই request-এ dependency-র ফল মনে রাখে, তাই দুজন **একই session** পায়।

**(গ) কেন এই উপায়, বিকল্প কী**
- **বিকল্প:** প্রতিটা endpoint-এ নিজে `db = SessionLocal()` লিখে শেষে `db.close()`। সমস্যা: ভুলে যাওয়া সহজ, আর মাঝে error হলে `close()` পর্যন্ত পৌঁছায় না। তখন connection খোলা থেকে যায়, pool ফুরিয়ে গেলে app আটকে যায়।
- **বিকল্প:** `with SessionLocal() as db:`। এটাও নিজে বন্ধ করে, কিন্তু FastAPI-র `Depends` দিয়ে `yield` রাখলে সব endpoint এক জায়গা থেকে session পায়। পরে test-এ নকল DB বসানোও সহজ (`app.dependency_overrides`)।

**(ঘ) Swagger-এ test**
- যে-কোনো DB endpoint (`/signup`, `/login`, ...) চালালেই `get_db` চলে। সফল হলে চাবি ঠিক আছে।

> **এক লাইনে:** `get_db` প্রতি request-এ চাবি দেয় (`yield`) আর যা-ই হোক ফেরত নেয় (`finally`)।

<a id="s10"></a>

### security.py: locker আর coupon

**গল্প (locker):** Locker-এর মালিক তোমার আসল চাবি রাখে না, শুধু চাবির একটা ছাপ রাখে। তুমি চাবি আনলে ছাপের সাথে মিলিয়ে দেখে। কেউ চুরি করে ছাপ পেলেও ছাপ থেকে চাবি বানাতে পারবে না।

**গল্প (coupon):** Login করলে একটা coupon পাও। তাতে লেখা কার coupon, কখন মেয়াদ শেষ, আর restaurant-এর গোপন সিল। সিল নকল করা যায় না, তাই guard শুধু সিল আর মেয়াদ দেখেই ঢুকতে দেয়।

**ধারণা:** Password সরাসরি না রেখে **bcrypt hash** রাখি। Login-এর পর একটা **JWT** দিই, যাতে `sub` (কে) আর `exp` (কখন মেয়াদ শেষ) থাকে, আর `SECRET_KEY` দিয়ে সই করা।

**(ক) আসল code**

```python
from passlib.context import CryptContext
import os
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from jose import jwt, JWTError


load_dotenv()

SECRET_KEY = os.getenv("SECRET_KEY")
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES= 30

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password):
    return pwd_context.hash(password)


def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.now(timezone.utc)+timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode["exp"]=expire
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def decode_access_token(token: str):
    try: 
        payload = jwt.decode(token, SECRET_KEY, algorithms=ALGORITHM)
        return payload["sub"]
    except JWTError:
        return None
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `SECRET_KEY = os.getenv("SECRET_KEY")` | সইয়ের চাবি `.env` থেকে। এটা জানলে যে-কেউ নকল token বানাতে পারবে |
| `ALGORITHM = "HS256"` | একই গোপন চাবি দিয়ে সই করা আর যাচাই করা |
| `ACCESS_TOKEN_EXPIRE_MINUTES= 30` | coupon-এর মেয়াদ 30 মিনিট |
| `CryptContext(schemes=["bcrypt"], deprecated="auto")` | bcrypt দিয়ে hash। পরে algorithm বদলালে পুরনো hash-কে "পুরনো" হিসেবে চিনতে পারে |
| `hash_password` | password → hash। একই password দুবার hash করলে **দুটো আলাদা** hash আসে (salt-এর কারণে) |
| `verify_password` | জমা hash থেকে salt বের করে, নতুন password-কে আবার hash করে মেলায় |
| `to_encode = data.copy()` | যে dict পাঠানো হয়েছে, সেটা না বদলে একটা কপিতে `exp` বসাই |
| `datetime.now(timezone.utc) + timedelta(...)` | এখন থেকে 30 মিনিট পর। সবসময় UTC, যাতে server-এর timezone-এ গোলমাল না হয় |
| `jwt.encode(...)` | payload + সই মিলিয়ে token string |
| `jwt.decode(...)` | সই মেলে কিনা **আর** `exp` পার হয়েছে কিনা, দুটোই নিজে থেকে দেখে |
| `return payload["sub"]` | token-এর মালিকের email |
| `except JWTError: return None` | সই ভুল, মেয়াদ শেষ, বা token ভাঙা, সব ক্ষেত্রেই `None`। পরে guard 401 দেয় |

**Salt কেন:** Salt হলো প্রতিটা password-এর সাথে মেশানো একটা random টুকরো। Salt ছাড়া দুজনের password `123456` হলে hash-ও এক হতো। তখন একটা ভাঙলেই অন্যটাও ভাঙা, আর আগে থেকে হিসাব করা hash-এর বড় তালিকা (rainbow table) দিয়ে সহজে মেলানো যেত। Salt থাকলে প্রতিটা hash আলাদা, তালিকা কাজ করে না। bcrypt salt-টা hash string-এর ভিতরেই রেখে দেয়, তাই আলাদা column লাগে না।

**bcrypt ইচ্ছা করে ধীর:** একটা hash বানাতে কিছুটা সময় লাগে। আমার login-এ সেটা টেরই পাওয়া যায় না, কিন্তু যে কোটি কোটি password চেষ্টা করছে তার জন্য অনেক ধীর।

**`exp` কেন:** মেয়াদ না থাকলে চুরি যাওয়া token চিরকাল কাজ করত। 30 মিনিট মানে চুরি হলেও ক্ষতির সময় সীমিত। দাম: user-কে 30 মিনিট পর আবার login করতে হয় (পরে refresh token দিয়ে এটা সহজ করা যায়)।

**JWT গোপন না:** Token-এর মাঝের অংশ (payload) শুধু base64, encrypt করা না। যে-কেউ পড়তে পারে, শুধু বদলাতে পারে না (বদলালে সই মিলবে না)। তাই payload-এ কখনো password বা গোপন কিছু রাখি না। শুধু `sub` আর `exp`।

**(গ) কেন এই উপায়, বিকল্প কী**
- **শুধু sha256 hash:** খুব দ্রুত আর salt নেই। Password-এর জন্য ভুল পছন্দ।
- **argon2:** bcrypt-এর চেয়ে নতুন, passlib-এও আছে। bcrypt এখনো নিরাপদ আর সবচেয়ে প্রচলিত।
- **Session cookie:** server নিজে মনে রাখে কে login করেছে। JWT-তে server কিছু মনে রাখে না, token-ই সব বলে দেয়।
- **PyJWT:** python-jose-এর বদলে আরেকটা JWT library, একই কাজ।
- **একটা ছোট নোট:** `jwt.decode`-এ `algorithms=ALGORITHM` দিয়েছি, অর্থাৎ একটা string। এখানে কাজ করছে, তবে library-র docs-এ list দেওয়া থাকে: `algorithms=[ALGORITHM]`।

**(ঘ) Swagger-এ test**
- `POST /signup` → সফল মানে `hash_password` চলেছে।
- `POST /login` ঠিক password দিয়ে → `access_token` পাবে (`create_access_token` চলেছে)। ভুল password → 401 (`verify_password` False দিয়েছে)।
- Token দিয়ে `GET /me` → 200 (`decode_access_token` চলেছে)। 30 মিনিট পর একই token → 401।

> **এক লাইনে:** Password-এর জায়গায় salt-মেশানো hash, আর login-এর প্রমাণ হিসেবে মেয়াদসহ সই-করা coupon (JWT)।


<a id="s11"></a>

### schemas.py: order form আর receipt

**গল্প:** Restaurant-এ order form-এ খদ্দের কী লিখতে পারবে তা ঠিক করা থাকে। Receipt-এ কী ছাপা হবে তাও ঠিক করা: খাবারের নাম আর দাম থাকে, রান্নাঘরের গোপন recipe থাকে না।

**ধারণা:** **Create** schema = যা client পাঠায় (input)। **Response** schema = যা client ফেরত পায় (output)। FastAPI দুটো দিয়েই data যাচাই আর ছাঁকাই করে।

**(ক) আসল code**

```python
from pydantic import BaseModel

class UserCreate(BaseModel):
    email: str
    password: str

class UserResponse(BaseModel):
    model_config = {"from_attributes": True}
    id: int
    email: str


class UserLogin(BaseModel):
    email: str
    password: str

class Token(BaseModel):
    access_token: str
    token_type: str

class ConversationCreate(BaseModel):
    title: str


class ConversationResponse(BaseModel):
    model_config = {"from_attributes": True}
    id: int
    title: str

class MessageCreate(BaseModel):
    content: str

class MessageResponse(BaseModel):
    model_config = {"from_attributes": True}
    id: int
    role: str
    content: str
```

**(খ) কোন লাইন কেন**

| Schema | কোথায় | কেন এই field গুলো |
|---|---|---|
| `UserCreate` | `/signup` input | email আর password। ভুল আকারে পাঠালে FastAPI নিজেই 422 দেয় |
| `UserResponse` | `/signup`, `/me` output | শুধু `id` আর `email`। **password বা hashed_password নেই** |
| `UserLogin` | `/login` input | এখন `UserCreate`-এর মতোই, কিন্তু পরে signup-এ নাম ইত্যাদি যোগ হলে login বদলাবে না |
| `Token` | `/login` output | `access_token` আর `token_type` |
| `ConversationCreate` | input | শুধু `title`। `user_id` নেই, কারণ সেটা token থেকে আসে |
| `ConversationResponse` | output | `id` আর `title` |
| `MessageCreate` | input | শুধু `content`। `role` নেই: client নিজেকে "assistant" বলে চালাতে পারবে না। `conversation_id` আসে URL থেকে |
| `MessageResponse` | output | `id`, `role`, `content` |
| `model_config = {"from_attributes": True}` | সব Response-এ | Pydantic সাধারণত dict পড়ে। এটা দিলে object-এর attribute (`user.id`, `user.email`) থেকেও পড়তে পারে। Endpoint SQLAlchemy object return করে, তাই এটা লাগে |

**কেন DB object কখনো `response_model` ছাড়া return করবো না:** `response_model` না দিলে FastAPI object-এর **সব column** JSON বানিয়ে পাঠায়। `User` object-এ `hashed_password` আছে, তাই সেটাও চলে যায়। আমার সাথে ঠিক এটাই হয়েছিল: [ভুল ১২](#m12)। `response_model` হলো ছাঁকনি: schema-তে যা নেই তা বাইরে যায় না।

**(গ) কেন এই উপায়, বিকল্প কী**
- **নিজে dict বানিয়ে return:** `return {"id": user.id, "email": user.email}`। কাজ করে, কিন্তু প্রতিটা endpoint-এ মনে রাখতে হয়। একবার ভুললেই leak।
- **SQLModel:** model আর schema এক class-এ। কম code, কিন্তু input/output আলাদা রাখার নিয়ম তখনো মানতে হয়।
- **`email: str`:** এখন যে-কোনো string চলে, `"abc"`-ও। পরে `EmailStr` দিয়ে আসল email যাচাই করবো (বাকি কাজের তালিকায় আছে)।

**(ঘ) Swagger-এ test**
- `/docs`-এর একদম নিচে **Schemas** অংশে সব schema দেখা যায়।
- `POST /signup`-এ শুধু `{"email": "test1@example.com"}` পাঠাও (password ছাড়া) → **422**, আর `detail`-এ লেখা থাকবে `password` field missing।
- সফল signup-এর response-এ শুধু `id` আর `email` আছে কিনা দেখো।

> **এক লাইনে:** Create = order form (ঢোকার দরজা), Response = receipt (বের হওয়ার ছাঁকনি)। Receipt ছাড়া কখনো DB object না।

<a id="s12"></a>

### llm.py: বাইরের রাঁধুনিকে ডাকা

**গল্প:** রাঁধুনির কোনো স্মৃতি নেই। তাই waiter প্রতিবার পুরো order-এর ইতিহাস কাগজে লিখে নিয়ে যায়। রাঁধুনি ব্যস্ত থাকলে ("এখন না", 429) বা রান্নাঘরে গোলমাল হলে (5xx) waiter দুই সেকেন্ড দাঁড়িয়ে আবার বলে, সব মিলিয়ে সর্বোচ্চ তিনবার।

**ধারণা:** `get_reply` পুরো history (message-এর list) Groq API-তে পাঠায় আর উত্তরের text ফেরত দেয়। সাময়িক সমস্যায় (429, 5xx) আবার চেষ্টা করে।

**(ক) আসল code**

```python
import os
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

URL = "https://api.groq.com/openai/v1/chat/completions"
HEADERS = {"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"}

def get_reply(messages, model="openai/gpt-oss-20b"):
    body = {"model": model, "messages": messages}

    for _ in range(3):
        response = httpx.post(URL, headers=HEADERS, json=body, timeout=30)
        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(2)
            continue
        break

    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]
```

**History list দেখতে এমন** (`main.py` এটা বানিয়ে পাঠায়):

```python
[
    {"role": "user", "content": "হ্যালো"},
    {"role": "assistant", "content": "হ্যালো! কীভাবে সাহায্য করতে পারি?"},
    {"role": "user", "content": "আমার আগের প্রশ্ন কী ছিল?"},
]
```

LLM নিজে কিছু মনে রাখে না (stateless)। আগের কথা মনে আছে বলে মনে হয় কারণ প্রতিবার পুরো list যায়।

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `URL = ".../openai/v1/chat/completions"` | Groq-এর OpenAI-র মতো endpoint। একই format, শুধু ঠিকানা আলাদা |
| `HEADERS = {"Authorization": f"Bearer {...}"}` | API key `.env` থেকে, header-এ Bearer হিসেবে |
| `def get_reply(messages, model="openai/gpt-oss-20b")` | model না বললে এটাই। পরে conversation অনুযায়ী model বদলানো যাবে |
| `body = {"model": ..., "messages": ...}` | API-র যা দরকার, শুধু তাই |
| `for _ in range(3)` | সর্বোচ্চ তিনবার চেষ্টা। `_` মানে গণনার নম্বর আমার লাগবে না |
| `timeout=30` | 30 সেকেন্ডে উত্তর না এলে হাল ছাড়ি, server চিরকাল আটকে থাকবে না |
| `status_code == 429 or status_code >= 500` | 429 = খুব বেশি request, 5xx = ওদের server-এর সমস্যা। দুটোই সাময়িক, একটু পর ঠিক হতে পারে |
| `time.sleep(2)` + `continue` | 2 সেকেন্ড অপেক্ষা করে আবার চেষ্টা |
| `break` | 200 (সফল) বা 400/401-এর মতো error। আবার চেষ্টা করে লাভ নেই, তাই loop থেকে বের হই |
| `response.raise_for_status()` | এখনো error থাকলে exception তুলি, যাতে ভুল উত্তর চুপচাপ পাস না হয় |
| `["choices"][0]["message"]["content"]` | উত্তরের JSON থেকে শুধু text বের করা |

**401 বা 400-এ retry কেন না:** key ভুল বা body ভুল হলে 100 বার চেষ্টা করলেও একই ফল। Retry শুধু সাময়িক সমস্যার জন্য।

**(গ) কেন এই উপায়, বিকল্প কী**
- **httpx সরাসরি বনাম openai SDK:** Week 08-এর `stream_server.py`-তে openai SDK (`base_url` দিয়ে Groq) ব্যবহার করেছিলাম। এখানে httpx দিয়ে নিচের আসল HTTP call দেখছি। SDK-তে retry আর আলাদা error type (যেমন `RateLimitError`) তৈরি থাকে।
- **Exponential backoff:** প্রতিবার একই 2 সেকেন্ড না, বরং 1, 2, 4 সেকেন্ড। আর 429-এর response-এ `Retry-After` header থাকলে সেটা মানা।
- **এখনকার সীমা:**
  - তিনবারই ব্যর্থ হলে `raise_for_status()` exception তোলে, আর user পায় **500**। এটা ঠিক করা বাকি কাজের তালিকায় আছে (friendly error)।
  - শেষ চেষ্টার পরেও 2 সেকেন্ড ঘুমায়, যদিও আর চেষ্টা হবে না।
  - Internet-ই না থাকলে বা timeout হলে httpx নিজেই exception তোলে। সেটা retry হয় না।

**(ঘ) Swagger-এ test**
- `llm.py`-র নিজের কোনো endpoint নেই। `POST /conversations/{id}/messages` চালালে এটা চলে। উত্তরে `"role": "assistant"` আর LLM-এর text এলে সফল।
- Groq-এ সমস্যা হলে Swagger-এ 500 আসবে, আর uvicorn terminal-এ `httpx.HTTPStatusError` দেখা যাবে ([500 debug](#s24) দেখো)।

> **এক লাইনে:** প্রতিবার পুরো history পাঠাই কারণ LLM-এর স্মৃতি নেই। সাময়িক error-এ (429/5xx) তিনবার পর্যন্ত চেষ্টা করি।


<a id="s13"></a>

### main.py: শুরুর অংশ (import, app, bearer_scheme)

**গল্প:** Restaurant খোলার আগে waiter সবার সাথে পরিচিত হয়: কোথায় চাবি, কোথায় locker, কোন form, কে রাঁধুনি। তারপর দরজা খোলে।

**ধারণা:** `main.py` বাকি সব file থেকে দরকারি জিনিস import করে, FastAPI app বানায়, আর token পড়ার জন্য `HTTPBearer` ঠিক করে।

**(ক) আসল code**

```python
from database import get_db
from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from llm import get_reply
from models import Conversation, Message, User
from schemas import (
    ConversationCreate,
    ConversationResponse,
    MessageCreate,
    MessageResponse,
    Token,
    UserCreate,
    UserLogin,
    UserResponse,
)
from security import (
    create_access_token,
    decode_access_token,
    hash_password,
    verify_password,
)
from sqlalchemy.orm import Session

app = FastAPI()

bearer_scheme = HTTPBearer()
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `from database import get_db` | প্রতি request-এ DB session |
| `Depends, FastAPI, HTTPException` | dependency, app, আর error ফেরত দেওয়ার উপায় |
| `HTTPAuthorizationCredentials, HTTPBearer` | header থেকে token পড়া |
| `from llm import get_reply` | LLM call |
| `from models import ...` | তিনটা table |
| `from schemas import (...)` | সব input/output schema। একটা বাদ পড়লে কী হয়: [ভুল ৩](#m3) |
| `from security import (...)` | hash আর token-এর চারটা function |
| `app = FastAPI()` | পুরো app। uvicorn `main:app` দিয়ে এটাকেই চালায় |
| `bearer_scheme = HTTPBearer()` | request-এর `Authorization: Bearer <token>` header পড়ে |

**`HTTPBearer` কী করে:**
- Header ঠিক থাকলে একটা `HTTPAuthorizationCredentials` object দেয়। এর ভিতরে `scheme` (`"Bearer"`) আর `credentials` (আসল token string)।
- Header না থাকলে endpoint-এ পৌঁছানোর আগেই নিজে থেকে error দেয়: `{"detail": "Not authenticated"}`। আমার FastAPI version-এ এটা **401** (পুরনো version-এ 403 দিত)।
- Swagger-এ উপরে ডানে **Authorize** 🔓 বোতাম আসে এর কারণেই।

**(গ) কেন এই উপায়, বিকল্প কী**
- **`OAuth2PasswordBearer`:** FastAPI tutorial-এ এটা বেশি দেখা যায়। এটা ধরে নেয় login form-data (`username`, `password`) নেয়। আমার `/login` JSON নেয়, তাই `HTTPBearer` সহজ: Swagger-এ শুধু token paste করলেই হয়।

**(ঘ) Swagger-এ test**
- `uvicorn main:app --reload` চালিয়ে `http://127.0.0.1:8000/docs` খোলো। সব endpoint-এর তালিকা আর Authorize বোতাম দেখা গেলে শুরুর অংশ ঠিক আছে।
- Import-এ ভুল থাকলে server চালুই হবে না, terminal-এ `NameError` বা `ImportError` দেখাবে।

**Swagger-এ token বসানোর নিয়ম** (পরের সব protected endpoint-এ লাগবে):
1. `POST /login` → response থেকে `access_token`-এর মান copy (quote ছাড়া)।
2. উপরে **Authorize** 🔓 → Value বাক্সে শুধু token paste (সামনে `Bearer ` লিখবো না, Swagger নিজে বসায়) → **Authorize** → **Close**।
3. এখন তালা বন্ধ 🔒 দেখাবে, আর প্রতিটা request-এ header নিজে থেকে যাবে।

> **এক লাইনে:** `main.py`-র শুরুতে সবাইকে ডেকে আনা হয়, আর `HTTPBearer` দরজায় দাঁড়িয়ে coupon হাতে নেয়।

<a id="s14"></a>

### GET /

**গল্প:** Restaurant-এর দরজায় "খোলা আছে" বোর্ড। ভিতরে ঢোকার আগেই বোঝা যায় দোকান চলছে কিনা।

**ধারণা:** সবচেয়ে সহজ endpoint। কোনো DB, কোনো token লাগে না। শুধু বলে app চলছে।

**(ক) আসল code**

```python
@app.get("/")
def home():
    return {"message": "Chat app running"}
```

**(খ) কোন লাইন কেন**
- `@app.get("/")`: root ঠিকানায় GET request এলে নিচের function চলবে।
- `return {...}`: dict return করলে FastAPI নিজেই JSON বানায়।

**(গ) কেন এই উপায়, বিকল্প কী**
- এটাকে **health check** বলে। পরে deploy করলে hosting service এই ঠিকানায় মাঝে মাঝে ঢুঁ মেরে দেখে app বেঁচে আছে কিনা। বড় app-এ এখানে DB-ও ping করা হয়, যাতে DB বন্ধ থাকলেও ধরা পড়ে।

**(ঘ) Swagger-এ test**
- `GET /` → **Try it out** → **Execute** → 200, body: `{"message": "Chat app running"}`।

> **এক লাইনে:** `GET /` হলো "দোকান খোলা" বোর্ড।

<a id="s15"></a>

### POST /signup

**গল্প:** নতুন খদ্দের নাম লেখাতে এলো। Waiter আগে খাতা দেখে: এই নামে কেউ আছে কিনা। না থাকলে চাবির ছাপ (hash) নিয়ে খাতায় তোলে, আর একটা receipt দেয় যাতে শুধু নম্বর আর email।

**ধারণা:** Body যাচাই → email আগে আছে কিনা → password hash → DB-তে save → `UserResponse` দিয়ে ছেঁকে ফেরত।

**(ক) আসল code**

```python
@app.post("/signup", response_model=UserResponse)
def signup(user: UserCreate, db: Session = Depends(get_db)):
    existing = db.query(User).filter(User.email == user.email).first()
    if existing:
        raise HTTPException(status_code=400, detail="Email ache re bhai age thekei")
    hashed = hash_password(user.password)
    new_user = User(email=user.email, hashed_password=hashed)
    db.add(new_user)
    db.commit()
    db.refresh(new_user)
    return new_user
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `response_model=UserResponse` | ফেরত শুধু `id` আর `email`। hash কখনো বাইরে না |
| `user: UserCreate` | body-কে `UserCreate` দিয়ে যাচাই। ভুল হলে function-এ ঢোকার আগেই 422 |
| `db: Session = Depends(get_db)` | এই request-এর DB session |
| `.filter(User.email == user.email).first()` | একই email-এর user খোঁজা। না পেলে `None` |
| `raise HTTPException(status_code=400, ...)` | email আগে থেকে আছে, তাই 400 Bad Request |
| `hashed = hash_password(user.password)` | আসল password এই লাইনের পরে আর কোথাও যায় না |
| `User(email=..., hashed_password=hashed)` | নতুন row-এর Python object। এখনো DB-তে না |
| `db.add(new_user)` | session-কে বলা "এটা save করতে হবে" |
| `db.commit()` | আসলে DB-তে লেখা। এটা ছাড়া কিছুই save হয় না ([ভুল ২](#m2)) |
| `db.refresh(new_user)` | DB-র বানানো `id` আর `created_at` object-এ টেনে আনা |
| `return new_user` | ORM object। `response_model` + `from_attributes` এটাকে `{"id", "email"}` বানায় |

**(গ) কেন এই উপায়, বিকল্প কী**
- **`unique=True` থাকার পরেও আগে check কেন:** check না করলে DB নিজে আটকাবে, কিন্তু তখন `IntegrityError` উঠে user পাবে 500। আগে check করলে পরিষ্কার 400 আর বোঝার মতো message।
- **একই সময়ে দুজন একই email দিলে:** দুজনেই check পার হতে পারে। তখন `unique=True` শেষ রক্ষা করে (দ্বিতীয়জন 500 পায়)। আরো পোক্ত উপায়: `try/except IntegrityError` দিয়ে সেটাকেও 400 বানানো।
- **Status 200 বনাম 201:** নতুন কিছু বানালে HTTP-র নিয়মে 201 Created বেশি মানানসই। `@app.post(..., status_code=201)` দিলেই হয়। এখন 200 দিচ্ছে, কাজে সমস্যা নেই।

**(ঘ) Swagger-এ test**
1. `POST /signup` → Try it out → body: `{"email": "test1@example.com", "password": "pass1234"}` → Execute → **200**, `{"id": 1, "email": "test1@example.com"}` (id যা-ই হোক)।
2. একই body আবার → **400**, `{"detail": "Email ache re bhai age thekei"}`।
3. `password` বাদ দিয়ে পাঠাও → **422**।
4. Response-এ `hashed_password` নেই, এটা নিজের চোখে দেখো।

> **এক লাইনে:** Signup = যাচাই, duplicate check, hash, save, তারপর ছাঁকা receipt।

<a id="s16"></a>

### POST /login

**গল্প:** খদ্দের বলে "আমি অমুক, এই আমার চাবি।" Locker-ওয়ালা ছাপের সাথে মেলায়। মিললে একটা coupon দেয়, যাতে নাম আর মেয়াদ লেখা, আর restaurant-এর সিল। না মিললে শুধু বলে "ভুল", **কোনটা** ভুল সেটা বলে না।

**ধারণা:** Email দিয়ে user খোঁজা → password যাচাই → JWT বানানো → `Token` আকারে ফেরত।

**(ক) আসল code**

```python
@app.post("/login", response_model=Token)
def login(user: UserLogin, db: Session = Depends(get_db)):
    db_user = db.query(User).filter(User.email == user.email).first()
    if not db_user or not verify_password(user.password, db_user.hashed_password):
        raise HTTPException(status_code=401, detail="abeh bhul email or pass dichosh")
    token = create_access_token({"sub": db_user.email})
    return {"access_token": token, "token_type": "bearer"}
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `response_model=Token` | ফেরত সবসময় `access_token` আর `token_type` |
| `user: UserLogin` | body: email আর password |
| `db_user = ...first()` | এই email-এর user, না থাকলে `None` |
| `if not db_user or not verify_password(...)` | user নেই **অথবা** password ভুল, দুটোতেই একই পথ |
| `or`-এর ক্রম | `not db_user` True হলে Python ডান দিক আর দেখেই না। তাই `None.hashed_password` পড়ার error হয় না |
| `status_code=401` | পরিচয় প্রমাণ হয়নি (Unauthorized) |
| `create_access_token({"sub": db_user.email})` | coupon-এ মালিক হিসেবে email |
| `return {"access_token": token, "token_type": "bearer"}` | `Token` schema-র সাথে মেলে। `"bearer"` বলে দেয় header-এ কীভাবে পাঠাতে হবে |

**(গ) কেন এই উপায়, বিকল্প কী**
- **একই message কেন:** "এই email নেই" আর "password ভুল" আলাদা বললে চোর email বসিয়ে বসিয়ে জেনে নিতে পারত কোন email-এ account আছে। একই message দিলে কিছুই জানা যায় না। ([Security notes](#s23))
- **`sub`-এ email বনাম id:** user email বদলালে পুরনো token-এর `sub` আর মিলবে না। `str(db_user.id)` রাখলে id কখনো বদলায় না। এখনকার জন্য email-ই চলে।
- **বিকল্প:** `OAuth2PasswordRequestForm` দিয়ে form-data login। তাহলে Swagger-এর Authorize বোতামেই username/password দিয়ে login করা যেত।
- **পরে লাগবে:** একই IP থেকে বারবার ভুল চেষ্টায় rate limit।

**(ঘ) Swagger-এ test**
1. `POST /login` → body: `{"email": "test1@example.com", "password": "pass1234"}` → **200**, `{"access_token": "<লম্বা token>", "token_type": "bearer"}`।
2. ভুল password → **401**, `{"detail": "abeh bhul email or pass dichosh"}`।
3. নেই এমন email → **একই 401, একই message**। এটাই চাই।
4. সফল token-টা copy করে **Authorize**-এ বসাও ([main.py-র শুরুর অংশ](#s13) দেখো)।

> **এক লাইনে:** Login = খোঁজো, মেলাও, coupon দাও। ভুল হলে শুধু "ভুল", কোনটা ভুল তা না।

<a id="s17"></a>

### get_current_user: দরজার guard

**গল্প:** VIP অংশের দরজায় guard দাঁড়িয়ে। Coupon না থাকলে ঢোকা নিষেধ। Coupon-এর সিল নকল বা মেয়াদ শেষ হলেও নিষেধ। Coupon ঠিক থাকলে খাতায় দেখে নেয় লোকটা এখনো সদস্য কিনা। তারপর ভিতরে পাঠায়, সাথে বলে দেয় "ইনি অমুক"।

**ধারণা:** এটা endpoint না, একটা **dependency**। যে endpoint `Depends(get_current_user)` চায়, তার আগে এটা চলে। সফল হলে লগ-ইন করা `User` object দেয়, না হলে 401।

**(ক) আসল code**

```python
def get_current_user(
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
    db: Session = Depends(get_db),
):
    email = decode_access_token(credentials.credentials)
    if email is None:
        raise HTTPException(
            status_code=401, detail="abeh tor token er meyed nai abar login kor"
        )
    user = db.query(User).filter(User.email == email).first()
    if user is None:
        raise HTTPException(status_code=401, detail="User not found")
    return user
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `credentials = Depends(bearer_scheme)` | `HTTPBearer` header থেকে token তুলে দেয়। header না থাকলে এখানেই 401 `Not authenticated` |
| `db = Depends(get_db)` | user খোঁজার জন্য session |
| `decode_access_token(credentials.credentials)` | `.credentials` = শুধু token string। পুরো object পাঠালে কী হয়: [ভুল ১০](#m10) |
| `if email is None` | token ভাঙা, সই ভুল, বা মেয়াদ শেষ → 401 |
| `user = ...first()` | token ঠিক, কিন্তু user এখনো DB-তে আছে কিনা |
| `if user is None` | token-এর মালিক মুছে গেছে → 401 |
| `return user` | পুরো `User` object। endpoint-গুলো `current_user.id` ব্যবহার করে |

**কোন ক্ষেত্রে কী ফেরত আসে:**

| অবস্থা | কে আটকায় | ফল |
|---|---|---|
| Header নেই | `HTTPBearer` | 401, `Not authenticated` |
| Token ভাঙা বা নকল | `decode_access_token` → `None` | 401, code-এর message |
| Token-এর মেয়াদ শেষ | `jwt.decode` নিজে `exp` দেখে → `None` | 401, code-এর message |
| Token ঠিক, user মুছে গেছে | DB query → `None` | 401, `User not found` |
| সব ঠিক | — | `User` object |

**(গ) কেন এই উপায়, বিকল্প কী**
- **Dependency কেন:** প্রতিটা protected endpoint-এ এই লাইনগুলো আলাদা করে লিখলে কোথাও একটা ভুলে যেতাম। এখন এক জায়গায় লেখা, সবাই `Depends` দিয়ে ব্যবহার করে।
- **প্রতিবার DB-তে খোঁজা কেন:** শুধু token বিশ্বাস করলে মুছে যাওয়া user-ও 30 মিনিট পর্যন্ত ঢুকতে পারত। দাম: প্রতি request-এ একটা ছোট query।
- **User id সবসময় এখান থেকে:** endpoint-গুলো কখনো client-এর পাঠানো user_id নেয় না, শুধু `current_user.id`। ([Security notes](#s23))

**(ঘ) Swagger-এ test** (এর নিজের endpoint নেই, `GET /me` দিয়ে test করি)
1. Authorize না করে `GET /me` → **401** `Not authenticated`।
2. Authorize-এ আজেবাজে লেখা (যেমন `abc`) বসিয়ে `GET /me` → **401**, `{"detail": "abeh tor token er meyed nai abar login kor"}`।
3. সঠিক token বসিয়ে → **200**।

> **এক লাইনে:** `get_current_user` হলো guard: coupon দেখে, যাচাই করে, তারপর "ইনিই সেই user" বলে ভিতরে পাঠায়।

<a id="s18"></a>

### GET /me

**গল্প:** ভিতরে ঢুকে জিজ্ঞেস করলে, "আমি কে হিসেবে ঢুকেছি?" Guard যা বলেছিল, waiter সেটাই receipt-এ লিখে দেয়।

**ধারণা:** Guard-এর দেওয়া `User` object ফেরত দেয়, `UserResponse` দিয়ে ছেঁকে।

**(ক) আসল code**

```python
@app.get("/me", response_model=UserResponse)
def me(current_user: str = Depends(get_current_user)):
    return current_user
```

**(খ) কোন লাইন কেন**
- `response_model=UserResponse`: **সবচেয়ে জরুরি অংশ।** এটা ছাড়া `hashed_password` বাইরে চলে যেত ([ভুল ১২](#m12))।
- `Depends(get_current_user)`: guard পার না হলে function-এ ঢোকাই হয় না।
- `return current_user`: কোনো নতুন query লাগে না, guard আগেই user এনে দিয়েছে।
- **ছোট খটকা:** type hint `current_user: str` লেখা, কিন্তু আসলে আসে একটা `User` object। Python type hint নিজে যাচাই করে না, তাই চলে যাচ্ছে। পড়ার সময় বিভ্রান্তি এড়াতে `User` লেখাই ঠিক, যেমন নিচের endpoint-গুলোতে আছে।

**(গ) কেন এই উপায়, বিকল্প কী**
- Frontend-এর জন্য খুব কাজের: page খুললেই `/me` ডেকে বোঝা যায় user লগ-ইন আছে কিনা আর কে।
- বিকল্প: token-এর payload frontend-এ নিজে পড়া। কিন্তু তাতে user এখনো আছে কিনা জানা যায় না, আর token নকল কিনা যাচাই হয় না।

**(ঘ) Swagger-এ test**
- Authorize করে `GET /me` → **200**, `{"id": 1, "email": "test1@example.com"}`। Response-এ `hashed_password` নেই, নিশ্চিত হও।

> **এক লাইনে:** `/me` = guard যাকে চিনেছে তার receipt, hash ছাড়া।

<a id="s19"></a>

### POST /conversations

**গল্প:** খদ্দের নতুন টেবিল চায়। টেবিলের নাম খদ্দের বলে, কিন্তু টেবিল **কার নামে** বুক হবে সেটা খদ্দের বলে না, guard-এর দেওয়া পরিচয় থেকে waiter নিজে লেখে।

**ধারণা:** `title` আসে body থেকে, `user_id` আসে token থেকে।

**(ক) আসল code**

```python
@app.post("/conversations", response_model=ConversationResponse)
def create_conversation(
    data: ConversationCreate,
    db: Session = Depends(get_db),
    current_user: User = Depends(get_current_user),
):
    conversation = Conversation(title=data.title, user_id=current_user.id)
    db.add(conversation)
    db.commit()
    db.refresh(conversation)
    return conversation
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `data: ConversationCreate` | body-তে শুধু `title` |
| `current_user: User = Depends(get_current_user)` | লগ-ইন ছাড়া conversation বানানো যাবে না |
| `user_id=current_user.id` | মালিক token থেকে, client থেকে কখনো না |
| `add` → `commit` → `refresh` | save, তারপর DB-র দেওয়া `id` আনা |
| `response_model=ConversationResponse` | ফেরত `id` আর `title` |

**(গ) কেন এই উপায়, বিকল্প কী**
- **`user_id` body-তে নিলে কী হতো:** যে-কেউ অন্যের `user_id` বসিয়ে তার নামে conversation বানাতে পারত। Token-এর পরিচয় নকল করা যায় না, body-র সংখ্যা যায়।
- **পরে:** title খালি রেখে প্রথম message থেকে নিজে title বানানো (বাকি কাজের তালিকায়)।

**(ঘ) Swagger-এ test**
- Authorize করে body: `{"title": "প্রথম chat"}` → **200**, `{"id": 1, "title": "প্রথম chat"}`। এই `id` পরের message endpoint-এ লাগবে।
- Authorize ছাড়া → **401**।

> **এক লাইনে:** Conversation-এর নাম দেয় client, মালিক ঠিক করে token।

<a id="s20"></a>

### GET /conversations

**গল্প:** "আমার টেবিলগুলো দেখাও।" Waiter শুধু ওই খদ্দেরের নামে বুক করা টেবিল দেখায়, পুরো restaurant-এর না।

**ধারণা:** `user_id == current_user.id` দিয়ে filter করে শুধু নিজের conversation-গুলো।

**(ক) আসল code**

```python
@app.get("/conversations", response_model=list[ConversationResponse])
def list_conversations(
    db: Session = Depends(get_db),
    current_user: User = Depends(get_current_user),
):
    conversations = (
        db.query(Conversation).filter(Conversation.user_id == current_user.id).all()
    )
    return conversations
```

**(খ) কোন লাইন কেন**
- `response_model=list[ConversationResponse]`: একটা না, একটা **list** ফেরত আসবে। প্রতিটা item ছাঁকা হবে।
- `.filter(Conversation.user_id == current_user.id)`: এই লাইন বাদ দিলে সবার conversation সবাই দেখত।
- `.all()`: সব মেলে এমন row-এর list। কিছু না থাকলে খালি list `[]`, error না।

**(গ) কেন এই উপায়, বিকল্প কী**
- **Pagination নেই:** একজনের হাজারটা conversation হলে সব একসাথে আসবে। পরে `limit` আর `offset` (বা `skip`) query parameter দিয়ে পাতা পাতা করে আনা যায়।
- **ক্রম নেই:** `order_by` দেওয়া নেই, তাই DB যে ক্রমে দেয়। নতুনটা আগে চাইলে `order_by(Conversation.created_at.desc())`।

**(ঘ) Swagger-এ test**
1. User A দিয়ে login করে দুটো conversation বানাও → `GET /conversations` → দুটো item।
2. User B signup + login করে Authorize-এ B-এর token বসাও → `GET /conversations` → `[]`। A-এর কিছুই দেখা যায় না।

> **এক লাইনে:** List সবসময় "আমার" দিয়ে filter করা।

<a id="s21"></a>

### POST /conversations/{id}/messages

**গল্প:** খদ্দের তার টেবিল থেকে order দেয়। Waiter আগে দেখে টেবিলটা সত্যিই তার কিনা। তারপর order খাতায় লেখে, এই টেবিলের পুরনো সব order সাথে নিয়ে রাঁধুনির কাছে যায় (কারণ রাঁধুনি কিছু মনে রাখে না), খাবার এলে সেটাও খাতায় লেখে, তারপর টেবিলে দেয়।

**ধারণা:** এই app-এর আসল কাজ এখানে। ছয়টা ধাপ:

1. Conversation এই user-এর কিনা (না হলে 404)
2. User-এর message DB-তে save
3. এই conversation-এর পুরো history আনা, `id` অনুযায়ী সাজিয়ে
4. History-কে `[{"role", "content"}, ...]` list বানানো
5. `get_reply()` দিয়ে LLM-এর উত্তর
6. উত্তর `assistant` role দিয়ে save, আর সেটাই ফেরত

**(ক) আসল code**

```python
@app.post("/conversations/{conversation_id}/messages", response_model=MessageResponse)
def send_message(
    conversation_id: int,
    data: MessageCreate,
    db: Session = Depends(get_db),
    current_user: User = Depends(get_current_user),
):
    conversation = (
        db.query(Conversation)
        .filter(
            Conversation.id == conversation_id, Conversation.user_id == current_user.id
        )
        .first()
    )
    if conversation is None:
        raise HTTPException(status_code=404, detail="Conversation not found")
    user_message = Message(
        conversation_id=conversation.id, role="user", content=data.content
    )
    db.add(user_message)
    db.commit()
    db.refresh(user_message)
    history = (
        db.query(Message)
        .filter(Message.conversation_id == conversation.id)
        .order_by(Message.id)
        .all()
    )
    messages = [{"role": m.role, "content": m.content} for m in history]
    reply = get_reply(messages)
    assistant_message = Message(
        conversation_id=conversation.id, role="assistant", content=reply
    )
    db.add(assistant_message)
    db.commit()
    db.refresh(assistant_message)
    return assistant_message
```

**(খ) কোন লাইন কেন**

| লাইন | কেন |
|---|---|
| `conversation_id: int` | URL-এর `{conversation_id}` থেকে। সংখ্যা না হলে FastAPI নিজেই 422 দেয় |
| `data: MessageCreate` | body-তে শুধু `content` |
| `.filter(Conversation.id == conversation_id, Conversation.user_id == current_user.id)` | **একই query-তে** id আর মালিক দুটোই মেলানো। দুটো শর্ত কমা দিয়ে দিলে মানে AND |
| `if conversation is None: 404` | conversation নেই, **অথবা** আছে কিন্তু অন্যের। দুটোই একই উত্তর |
| `Message(..., role="user", ...)` | role সবসময় server বসায় |
| প্রথম `commit()` + `refresh()` | LLM call-এর **আগে** user-এর message save হয়ে যায় |
| `.order_by(Message.id)` | DB-কে ক্রম বলে না দিলে ক্রমের কোনো নিশ্চয়তা নেই। `id` বাড়তেই থাকে, তাই এটাই আসল ক্রম |
| `.all()` | এই মাত্র save করা message-ও এই list-এ আছে, কারণ query হচ্ছে commit-এর পরে |
| `[{"role": m.role, "content": m.content} for m in history]` | DB object থেকে API-র চাওয়া dict-এ রূপান্তর। `id`, `created_at` LLM-এর দরকার নেই |
| `reply = get_reply(messages)` | Groq-এ call। এখানেই সবচেয়ে বেশি সময় যায় |
| `Message(..., role="assistant", content=reply)` | উত্তরও history-র অংশ, তাই save |
| `return assistant_message` | client শুধু নতুন উত্তরটা পায় |

**(গ) কেন এই উপায়, বিকল্প কী**
- **History server-এ কেন, client-এ না:** client history পাঠালে নিজের মতো বানিয়ে ভুয়া "assistant" message ঢোকাতে পারত। আর browser বন্ধ করলে সব হারাত। DB-তে থাকলে যে-কোনো device থেকে চালিয়ে যাওয়া যায়।
- **User message আগে commit কেন:** LLM ব্যর্থ হলেও user-এর লেখা হারায় না। উল্টো দিক: তখন DB-তে একটা উত্তরহীন user message থেকে যায়, আর পরের বার পরপর দুটো user message যায়। এটা চলে, তবে friendly error-এর কাজ করার সময় এটা মাথায় রাখবো।
- **System prompt নেই:** list-এর শুরুতে `{"role": "system", "content": "..."}` দিলে LLM-এর আচরণ ঠিক করে দেওয়া যেত। Conversation অনুযায়ী model config-এর কাজে এটা আসবে।
- **খরচ বাড়তেই থাকে:** প্রতিটা নতুন message-এ পুরো history আবার যায়। লম্বা conversation-এ token খরচ বাড়ে, আর একসময় context window ছাড়িয়ে যাবে। তাই history ছোট করা (trimming) বাকি কাজে আছে।
- **Streaming নেই:** user পুরো উত্তর তৈরি হওয়া পর্যন্ত অপেক্ষা করে। Week 08-এর SSE দিয়ে এটা বদলাবো।

**(ঘ) Swagger-এ test**
1. Authorize করো, একটা conversation বানাও (ধরো `id` = 1)।
2. `POST /conversations/{conversation_id}/messages` → `conversation_id` = 1, body: `{"content": "হ্যালো, তুমি কে?"}` → **200**, `{"id": 2, "role": "assistant", "content": "..."}`।
3. আবার পাঠাও: `{"content": "আমি একটু আগে কী জিজ্ঞেস করেছিলাম?"}` → উত্তরে আগের প্রশ্নের কথা থাকলে history কাজ করছে।
4. `conversation_id` = 9999 → **404** `Conversation not found`।
5. User B-এর token দিয়ে A-এর conversation-এ পাঠাও → **404**, 403 না।

> **এক লাইনে:** মালিক যাচাই → save → পুরো history → LLM → উত্তর save → ফেরত।


<a id="s22"></a>

### GET /conversations/{id}/messages

**গল্প:** "এই টেবিলের সব order-এর হিসাব দেখাও।" Waiter আবারো আগে দেখে টেবিলটা তার কিনা, তারপর খাতা খুলে সময়ের ক্রমে সব দেখায়।

**ধারণা:** একই ownership check, তারপর conversation-এর সব message `id`-এর ক্রমে।

**(ক) আসল code**

```python
@app.get(
    "/conversations/{conversation_id}/messages", response_model=list[MessageResponse]
)
def get_messages(
    conversation_id: int,
    db: Session = Depends(get_db),
    current_user: User = Depends(get_current_user),
):
    conversation = (
        db.query(Conversation)
        .filter(
            Conversation.id == conversation_id,
            Conversation.user_id == current_user.id,
        )
        .first()
    )
    if conversation is None:
        raise HTTPException(status_code=404, detail="Conversation not found")
    messages = (
        db.query(Message)
        .filter(Message.conversation_id == conversation.id)
        .order_by(Message.id)
        .all()
    )
    return messages
```

**(খ) কোন লাইন কেন**
- `response_model=list[MessageResponse]`: প্রতিটা message-এর `id`, `role`, `content`।
- Conversation query: ঠিক আগের endpoint-এর মতো, id আর `user_id` একসাথে। এটা বাদ দিলে যে-কেউ id বদলে বদলে অন্যের chat পড়তে পারত।
- `.order_by(Message.id)`: চ্যাট উপর থেকে নিচে সঠিক ক্রমে।

**(গ) কেন এই উপায়, বিকল্প কী**
- **একই check দুবার লেখা:** দুটো endpoint-এ একই conversation query। আরো endpoint (যেমন delete) এলে এটাকে একটা ছোট function বা dependency বানালে ভালো, যেমন `get_owned_conversation(conversation_id, db, current_user)`।
- **Pagination:** খুব লম্বা chat-এ শেষের 50টা আগে আনা, বাকিটা scroll করলে।

**(ঘ) Swagger-এ test**
1. আগের endpoint-এ দুটো message পাঠানোর পর `GET /conversations/1/messages` → **200**, 4টা item: user, assistant, user, assistant, এই ক্রমে।
2. অন্য user-এর token দিয়ে → **404**।

> **এক লাইনে:** History পড়ার আগেও একই প্রশ্ন: "এটা কি তোমার?"

<a id="p4"></a>

## ৪. Security notes

<a id="s23"></a>

### চারটা নিয়ম যা ভাঙা যাবে না

**গল্প:** ভালো guard চোরকে কোনো সূত্র দেয় না। "এই নামে কেউ নেই" বা "এই ঘরটা অন্যের" বললেও চোর কিছু শিখে যায়।

**১. ভুল email আর ভুল password-এ একই message**
- কোথায়: `POST /login`, এক `if`-এ দুটো শর্ত: `if not db_user or not verify_password(...)`।
- কেন: আলাদা message দিলে চোর হাজারটা email চেষ্টা করে জেনে নেয় কোনগুলোতে account আছে (account enumeration)। তারপর শুধু সেগুলোর password-এ আক্রমণ করে।

**২. user_id সবসময় token থেকে, client থেকে কখনো না**
- কোথায়: সব জায়গায় `current_user.id`। কোনো input schema-তে `user_id` field নেই।
- কেন: Body, URL বা query-র যে-কোনো সংখ্যা client বদলাতে পারে। Token `SECRET_KEY` দিয়ে সই করা, ওটা বদলালে সই মেলে না।

**৩. Ownership check: 403 না, 404**
- কোথায়: দুটো message endpoint-এ `Conversation.id == conversation_id, Conversation.user_id == current_user.id` একই query-তে।
- কেন: 403 Forbidden মানে "আছে, কিন্তু তোমার না"। এতে অন্তত জানা যায় ওই id-র conversation আছে। 404 মানে "পাওয়া যায়নি"। "নেই" আর "অন্যের", দুটো বাইরে থেকে একই রকম দেখায়, তাই কোনো তথ্য ফাঁস হয় না।
- একই query-তে দুটো শর্ত দেওয়ায় code-ও সহজ: আলাদা করে "আছে কিনা" আর "মালিক কিনা" দেখতে হয় না।

**৪. hashed_password কখনো response-এ না**
- কোথায়: `/signup` আর `/me`-তে `response_model=UserResponse`, যাতে শুধু `id` আর `email`।
- কেন: hash থেকে সরাসরি password পাওয়া যায় না, কিন্তু hash হাতে পেলে চোর নিজের computer-এ বসে অসংখ্য password চেষ্টা করে মেলাতে পারে (offline attack)। Hash-ও গোপন জিনিস।

**আরো কয়েকটা কথা**
- `SECRET_KEY` লম্বা আর random হওয়া চাই। ছোট বা অনুমানযোগ্য হলে কেউ নিজে token বানিয়ে ফেলতে পারে।
- Deploy করার পর HTTPS ছাড়া চলবে না। Token header-এ যায়, HTTP-তে পথেই কেউ পড়ে ফেলতে পারে।
- Error message-এ কখনো stack trace, SQL বা key বাইরে পাঠাবো না। FastAPI 500-এ শুধু `Internal Server Error` বলে, বিস্তারিত থাকে terminal-এ। এটাই ঠিক।

> **এক লাইনে:** চোরকে কোনো সূত্র দিও না: একই error, token-এর পরিচয়, 404, আর hash কখনো বাইরে না।

<a id="p5"></a>

## ৫. বানাতে গিয়ে যে ভুলগুলো করেছি

প্রতিটার জন্য: কী লিখেছিলাম, কী error এসেছিল, কীভাবে ঠিক করলাম, আর কেন হয়েছিল।
এই ভুলগুলোই সবচেয়ে ভালো শিক্ষক। আবার একই error দেখলে এখানে এসে মিলিয়ে নেবো।

<a id="m1"></a>

### ভুল ১: `=` বনাম `:` (existing:)

**কী লিখেছিলাম (ভুল):**

```python
existing: db.query(User).filter(User.email == user.email).first()
if existing:
```

**Error:**

```
UnboundLocalError: cannot access local variable 'existing' where it is not associated with a value
```

**ঠিক করলাম:**

```python
existing = db.query(User).filter(User.email == user.email).first()
if existing:
```

**কেন হলো:** `:` দিলে Python এটাকে type annotation (শুধু লেবেল) ধরে, কোনো মান বসায় না। তাই পরের লাইনে `existing` পড়তে গেলে মান নেই। মান বসাতে `=`। আর উল্টো দিকে, `if existing` লিখে শেষে `:` ভুলে গেলে আসে `SyntaxError: expected ':'`।

> **এক লাইনে:** `=` মান বসায়, `:` শুধু লেবেল দেয় (আর if/def-এর শেষে বসে)।

<a id="m2"></a>

### ভুল ২: `db.commit`-এ () নেই

**কী লিখেছিলাম (ভুল):**

```python
db.add(new_user)
db.commit
db.refresh(new_user)
```

**Error:**

```
sqlalchemy.exc.InvalidRequestError: Instance '<User at 0x...>' is not persistent within this Session
```

**ঠিক করলাম:**

```python
db.add(new_user)
db.commit()
db.refresh(new_user)
```

**কেন হলো:** `db.commit` শুধু function-টার নাম নেয়, **ডাকে না**। Python এতে কোনো error দেয় না, তাই লাইনটা চুপচাপ কিছু না করে পার হয়। কিছুই save হয়নি, তাই `refresh` বলে এই object DB-তে নেই।

> **এক লাইনে:** Function চালাতে () লাগবেই। () ছাড়া শুধু নাম।

<a id="m3"></a>

### ভুল ৩: অন্য file থেকে import ভুলে যাওয়া

**কী লিখেছিলাম (ভুল):**

```python
# main.py: hash_password বাদ পড়েছে
from security import create_access_token, decode_access_token, verify_password

# main.py: ConversationResponse বাদ পড়েছে
from schemas import ConversationCreate, MessageCreate, MessageResponse, Token, UserCreate, UserLogin, UserResponse
```

**Error:**

```
# /signup চালালে (server চালু ছিল, request-এ 500):
NameError: name 'hash_password' is not defined

# server চালু করার সময়েই (uvicorn উঠলই না):
NameError: name 'ConversationResponse' is not defined
```

**ঠিক করলাম:**

```python
from schemas import (
    ConversationCreate,
    ConversationResponse,
    ...
)
from security import (
    create_access_token,
    decode_access_token,
    hash_password,
    verify_password,
)
```

**কেন হলো:** এক file-এ লেখা function বা class অন্য file আপনা থেকে চেনে না, import করতে হয়। মজার পার্থক্য: `hash_password` function-এর **ভিতরে** ব্যবহার হয়, তাই error আসে request-এর সময়। `ConversationResponse` decorator-এ (`response_model=...`) ব্যবহার হয়, যা file load হওয়ার সময়েই চলে, তাই server চালুই হয় না। (ঠিক করা code-এ `...` মানে বাকি নামগুলো, যেগুলো আগের মতোই আছে।)

> **এক লাইনে:** অন্য file-এর নাম ব্যবহার করলে import-এ যোগ করো। Function-এর ভিতরের ভুল ধরা পড়ে request-এ, decorator-এর ভুল ধরা পড়ে startup-এ।

<a id="m4"></a>

### ভুল ৪: code-এ `...` রয়ে গেছে (Ellipsis)

**কী লিখেছিলাম (ভুল):**

```python
token = ...
return {"access_token": token, "token_type": "bearer"}
```

**Error:**

```
fastapi.exceptions.ResponseValidationError: 1 validation error:
  {'type': 'string_type', 'loc': ('response', 'access_token'), 'msg': 'Input should be a valid string', 'input': Ellipsis}
```

**ঠিক করলাম:**

```python
token = create_access_token({"sub": db_user.email})
return {"access_token": token, "token_type": "bearer"}
```

**কেন হলো:** লেখার সময় "পরে লিখবো" ভেবে `...` বসিয়েছিলাম। কিন্তু `...` Python-এ একটা আসল মান (`Ellipsis`), তাই Python কোনো আপত্তি করে না। শেষে response যাচাইয়ের সময় ধরা পড়ে, কারণ `access_token` string হওয়ার কথা।

> **এক লাইনে:** Run করার আগে Ctrl+F দিয়ে `...` খুঁজে দেখো। Error-এ `Ellipsis` দেখলেই বুঝবে কোথাও `...` রয়ে গেছে।

<a id="m5"></a>

### ভুল ৫: `in None` বনাম `is None`

**কী লিখেছিলাম (ভুল):**

```python
email = decode_access_token(credentials.credentials)
if email in None:
```

**Error:**

```
TypeError: argument of type 'NoneType' is not iterable
```

**ঠিক করলাম:**

```python
email = decode_access_token(credentials.credentials)
if email is None:
```

**কেন হলো:** `in` জিজ্ঞেস করে "এটা কি ওই list/string-এর **ভিতরে** আছে?" `None`-এর ভিতরে কিছু থাকে না, তাই Python ভিতরে খুঁজতেই পারে না। `is` জিজ্ঞেস করে "এটা কি **ঠিক সেই** জিনিস?"

> **এক লাইনে:** `None` দেখতে সবসময় `is None` বা `is not None`।

<a id="m6"></a>

### ভুল ৬: `.filter()` আছে, `.first()` নেই

**কী লিখেছিলাম (ভুল):**

```python
db_user = db.query(User).filter(User.email == user.email)
if not db_user or not verify_password(user.password, db_user.hashed_password):
```

**Error:**

```
AttributeError: 'Query' object has no attribute 'hashed_password'
```

**ঠিক করলাম:**

```python
db_user = db.query(User).filter(User.email == user.email).first()
if not db_user or not verify_password(user.password, db_user.hashed_password):
```

**কেন হলো:** `.filter()` শুধু query **লেখে**, চালায় না। হাতে থাকে একটা `Query` object, কোনো user না। `.first()` query চালিয়ে প্রথম row (বা `None`) দেয়। আরেকটা লুকানো বিপদ: `Query` object সবসময় True ধরা হয়, তাই `if not db_user` কখনো কাজ করত না।

> **এক লাইনে:** `.filter()` = order লেখা, `.first()` / `.all()` = order দিয়ে জিনিস আনা।

<a id="m7"></a>

### ভুল ৭: `return`-এর পরে লাইন

**কী লিখেছিলাম (ভুল):**

```python
db.add(new_user)
return new_user
db.commit()
db.refresh(new_user)
```

**Error:**

```
fastapi.exceptions.ResponseValidationError: 1 validation error:
  {'type': 'int_type', 'loc': ('response', 'id'), 'msg': 'Input should be a valid integer', 'input': None}
```

**ঠিক করলাম:**

```python
db.add(new_user)
db.commit()
db.refresh(new_user)
return new_user
```

**কেন হলো:** `return`-এ function সাথে সাথে শেষ। নিচের `commit` আর `refresh` কখনো চলে না। তাই user save হয়নি, `id` পায়নি (`None`), আর `UserResponse` বলে `id` integer হতে হবে। VS Code এই লাইনগুলো ঝাপসা করে দেখায় (unreachable code), ওটা দেখেই সন্দেহ করা উচিত ছিল।

> **এক লাইনে:** `return` সবসময় শেষ লাইন। তার নিচে কিছু থাকলে সেটা কখনো চলবে না।

<a id="m8"></a>

### ভুল ৮: Variable দিয়ে class ঢেকে দেওয়া (Token = ...)

**কী লিখেছিলাম (ভুল):**

```python
Token = create_access_token({"sub": db_user.email})
return Token(access_token=Token, token_type="bearer")
```

**Error:**

```
TypeError: 'str' object is not callable
```

**ঠিক করলাম:**

```python
token = create_access_token({"sub": db_user.email})
return {"access_token": token, "token_type": "bearer"}
```

**কেন হলো:** `Token` নামটা `schemas.py`-র class-এর। একই নামে string বসাতেই এই function-এর ভিতরে `Token` আর class না, একটা string। তারপর `Token(...)` মানে string-কে function-এর মতো ডাকা।

> **এক লাইনে:** Variable ছোট হাতের (`token`), class বড় হাতের (`Token`)। কখনো একই নাম না।

<a id="m9"></a>

### ভুল ৯: env variable-এর নাম মেলেনি (SECRET বনাম SECRET_KEY)

**কী লিখেছিলাম (ভুল):**

```python
# .env-এ নাম SECRET_KEY, কিন্তু security.py-তে:
SECRET_KEY = os.getenv("SECRET")
```

**Error:**

```
# /login চালালে 500, terminal-এ:
jose.exceptions.JWSError: Expecting a string- or bytes-formatted key.
```

**ঠিক করলাম:**

```python
SECRET_KEY = os.getenv("SECRET_KEY")
```

**কেন হলো:** `os.getenv` নাম না পেলে কোনো error দেয় না, চুপচাপ `None` দেয়। তাই server ঠিকই চালু হয়, সমস্যা ধরা পড়ে অনেক পরে, যখন `None` দিয়ে token সই করতে যায়। বিকল্প: `os.environ["SECRET_KEY"]` লিখলে নাম না পেলে প্রথমেই `KeyError` দিয়ে থেমে যায় (Week 08-এর `stream_server.py`-তে এভাবেই লিখেছিলাম)।

> **এক লাইনে:** `.env`-এর নাম আর `getenv`-এর নাম অক্ষরে অক্ষরে এক হতে হবে। Error-এ key নিয়ে অভিযোগ মানে আগে নাম মেলাও।

<a id="m10"></a>

### ভুল ১০: `credentials` বনাম `credentials.credentials`

**কী লিখেছিলাম (ভুল):**

```python
email = decode_access_token(credentials)
```

**Error:**

```
# GET /me → 500 (401 না!), terminal-এ:
AttributeError: 'HTTPAuthorizationCredentials' object has no attribute 'rsplit'
```

**ঠিক করলাম:**

```python
email = decode_access_token(credentials.credentials)
```

**কেন হলো:** `HTTPBearer` একটা object দেয়, যার ভিতরে দুটো জিনিস: `scheme` (`"Bearer"`) আর `credentials` (আসল token string)। পুরো object পাঠানোয় library string-এর কাজ (`rsplit`) করতে পারেনি। আর এই error `JWTError` না, তাই `except JWTError` ধরতে পারেনি, ফলে 401-এর বদলে 500।

> **এক লাইনে:** Object-এর ভিতর থেকে ঠিক জিনিসটা বের করো: `credentials.credentials` = token string।

<a id="m11"></a>

### ভুল ১১: token-এর লাইন failure-এর if-এর ভিতরে

**কী লিখেছিলাম (ভুল):**

```python
if not db_user or not verify_password(user.password, db_user.hashed_password):
    raise HTTPException(status_code=401, detail="abeh bhul email or pass dichosh")
    token = create_access_token({"sub": db_user.email})
    return {"access_token": token, "token_type": "bearer"}
```

**Error:**

```
# ঠিক password দিলে 500, terminal-এ:
fastapi.exceptions.ResponseValidationError: 1 validation error:
  {'type': 'model_attributes_type', 'loc': ('response',), 'msg': 'Input should be a valid dictionary or object to extract fields from', 'input': None}
```

**ঠিক করলাম:**

```python
if not db_user or not verify_password(user.password, db_user.hashed_password):
    raise HTTPException(status_code=401, detail="abeh bhul email or pass dichosh")
token = create_access_token({"sub": db_user.email})
return {"access_token": token, "token_type": "bearer"}
```

**কেন হলো:** Python-এ indent (সামনের ফাঁকা জায়গা) ঠিক করে কোন লাইন কার ভিতরে। Token-এর লাইন দুটো `if`-এর ভিতরে, `raise`-এর নিচে, তাই কখনো চলে না। Password ঠিক হলে `if`-এ ঢোকেই না, function কিছু return না করে শেষ হয়, মানে `None`। `Token` schema `None` নিতে পারে না।

> **এক লাইনে:** সফল পথের code `if`-এর বাইরে, বাম দিকে এক ধাপ সরানো।

<a id="m12"></a>

### ভুল ১২: response_model ছাড়া /me থেকে hash বেরিয়ে গেল

**কী লিখেছিলাম (ভুল):**

```python
@app.get("/me")
def me(current_user: str = Depends(get_current_user)):
    return current_user
```

**Error:**

```
# কোনো error নেই! এটাই বিপদ। Swagger-এ response:
{
  "id": 1,
  "email": "test1@example.com",
  "hashed_password": "<এখানে পুরো bcrypt hash চলে আসছিল>"
}
```

**ঠিক করলাম:**

```python
@app.get("/me", response_model=UserResponse)
def me(current_user: str = Depends(get_current_user)):
    return current_user
```

**কেন হলো:** `response_model` না থাকলে FastAPI SQLAlchemy object-এর সব column JSON বানিয়ে পাঠায়। `User`-এ `hashed_password` আছে, তাই সেটাও গেছে। কোনো error আসে না, তাই চোখে না দেখলে ধরা পড়ে না।

> **এক লাইনে:** DB object return করলে সবসময় `response_model` দাও। প্রতিটা response নিজের চোখে একবার দেখো।

<a id="m13"></a>

### ভুল ১৩: Docker Desktop বন্ধ, তাই chat-db বন্ধ

**কী লিখেছিলাম (ভুল):**

```
# কোনো code ভুল না। Computer restart-এর পর সরাসরি:
uvicorn main:app --reload
# তারপর Swagger-এ POST /signup
```

**Error:**

```
# Swagger-এ 500, terminal-এ:
sqlalchemy.exc.OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5434 failed: Connection refused
```

**ঠিক করলাম:**

```
# ১. Docker Desktop চালু করো (চালু হওয়া পর্যন্ত অপেক্ষা)
# ২. container চালু:
docker start chat-db
# ৩. যাচাই: chat-db-র STATUS "Up" কিনা
docker ps
```

**কেন হলো:** Computer restart হলে Docker Desktop নিজে চালু না হলে container-ও বন্ধ থাকে। Port 5434-এ কেউ শুনছে না, তাই connection refused। `docker update --restart unless-stopped chat-db` দিলে Docker চালু হলেই container নিজে চালু হবে।

> **এক লাইনে:** `Connection refused` + port 5434 মানে code না, গুদামের দরজা বন্ধ: আগে Docker Desktop, তারপর `docker start chat-db`।

<a id="p6"></a>

## ৬. 500 error কীভাবে debug করবো

<a id="s24"></a>

### Traceback পড়ার নিয়ম

**গল্প:** রোগী বলে "অসুস্থ লাগছে" (Swagger বলে `Internal Server Error`)। ডাক্তার এতে কিছু বোঝে না, report দেখে। আমার report হলো **uvicorn চলা terminal**। Swagger কখনো বিস্তারিত বলবে না, ইচ্ছা করেই (security)।

**ধারণা:** Traceback উপর থেকে নিচে ঘটনার ক্রম। কিন্তু পড়তে হয় **নিচ থেকে উপরে**:

1. **Terminal-এ যাও**, যেখানে `uvicorn main:app --reload` চলছে। লাল লেখার শেষ অংশে scroll করো।
2. **শেষ লাইন আগে পড়ো।** এটা error-এর type আর message: `NameError: ...`, `AttributeError: ...`। অর্ধেক সমস্যা এখানেই বোঝা যায়।
3. **উপরে উঠে নিজের file খোঁজো।** `site-packages`-এর লাইনগুলো library-র, ওগুলো বাদ। শেষ থেকে উপরে উঠতে প্রথম যেখানে `File "...\app\main.py", line N` (বা `llm.py`, `security.py`) দেখবে, সেটাই আমার code-এর জায়গা।
4. **সেই file-এর ওই line খোলো।** VS Code-এ `Ctrl+G` চেপে line number দিলে সরাসরি যায়।
5. **ঠিক করে save করো।** `--reload` নিজেই server আবার চালু করে। Swagger-এ আবার চেষ্টা।

**উদাহরণ:** Groq বারবার 429 দিলে `POST /conversations/1/messages`-এ এরকম traceback আসে (মাঝের library-র লাইন ছেঁটে দেওয়া):

```
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  ...
  File "D:\AI-Roadmap\Phase_02\chat_app\app\main.py", line 129, in send_message
    reply = get_reply(messages)
  File "D:\AI-Roadmap\Phase_02\chat_app\app\llm.py", line 22, in get_reply
    response.raise_for_status()
  File "...\site-packages\httpx\_models.py", line ..., in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '429 Too Many Requests' for url 'https://api.groq.com/openai/v1/chat/completions'
```

পড়ার ক্রম:
- **শেষ লাইন:** `httpx.HTTPStatusError ... 429` → Groq বলছে খুব বেশি request।
- **উপরে নিজের শেষ file:** `llm.py`, line 22: `raise_for_status()`, মানে তিনবার চেষ্টার পরেও 429।
- **তার উপরে:** `main.py`, line 129: `send_message`-এর `get_reply` থেকে এসেছে।
- **সমাধান:** একটু পরে আবার চেষ্টা। স্থায়ী সমাধান বাকি কাজের "friendly error"।

**শেষ লাইন দেখে দ্রুত অনুমান:**

| শেষ লাইনে | সম্ভবত | দেখো |
|---|---|---|
| `NameError: name '...' is not defined` | import বা বানান ভুল | [ভুল ৩](#m3) |
| `AttributeError: 'Query' object ...` | `.first()` বাদ | [ভুল ৬](#m6) |
| `AttributeError: 'NoneType' object has no attribute ...` | `None` হতে পারে এমন কিছু check না করে ব্যবহার | `is None` check আছে কিনা |
| `TypeError: argument of type 'NoneType' is not iterable` | `in None` | [ভুল ৫](#m5) |
| `TypeError: '...' object is not callable` | নাম দিয়ে class ঢেকে দেওয়া | [ভুল ৮](#m8) |
| `ResponseValidationError` | return করা জিনিস `response_model`-এর সাথে মেলে না | [ভুল ৪](#m4), [৭](#m7), [১১](#m11) |
| `InvalidRequestError ... not persistent` | `commit` চলেনি | [ভুল ২](#m2) |
| `OperationalError ... Connection refused` | DB বন্ধ | [ভুল ১৩](#m13) |
| `JWSError: Expecting a string- or bytes-formatted key` | `SECRET_KEY` পাওয়া যায়নি | [ভুল ৯](#m9) |
| `httpx.HTTPStatusError` | LLM API-র সমস্যা (key, limit, server) | `llm.py` |

**422 আর 500 আলাদা:** 422 মানে client ভুল body পাঠিয়েছে। বিস্তারিত Swagger-এর response-এই থাকে (কোন field, কী সমস্যা), terminal দেখতে হয় না। 500 মানে আমার code-এ সমস্যা, বিস্তারিত শুধু terminal-এ।

> **এক লাইনে:** 500 পেলে terminal → শেষ লাইন → উপরে নিজের file-এর line N → সেই লাইন ঠিক করো।

<a id="p7"></a>

## ৭. নিজেকে যাচাই

<a id="s25"></a>

### ১০টা প্রশ্ন (উত্তর লুকানো)

আগে নিজে উত্তর ভাবো, তারপর "উত্তর"-এ চেপে মিলিয়ে নাও।

**১. Password সরাসরি না রেখে hash কেন রাখি? Salt কী কাজ করে?**

<details><summary>উত্তর</summary>

DB চুরি হলেও আসল password পাওয়া যায় না। Salt প্রতিটা password-এ random টুকরো মেশায়, তাই একই password-এর hash-ও আলাদা হয়, আর আগে থেকে বানানো hash-তালিকা (rainbow table) কাজ করে না।

</details>

**২. `get_db`-তে `return` না দিয়ে `yield` কেন? `finally` না থাকলে কী হতো?**

<details><summary>উত্তর</summary>

`yield` থেমে থাকে, endpoint শেষ হলে ফিরে এসে `close()` করতে পারে। `return`-এ সেই সুযোগ নেই। `finally` না থাকলে error হলে session বন্ধ হতো না, connection জমতে জমতে pool ফুরিয়ে app আটকে যেত।

</details>

**৩. Login-এ ভুল email আর ভুল password-এ একই message কেন?**

<details><summary>উত্তর</summary>

আলাদা message দিলে কেউ বুঝে যেত কোন email-এ account আছে (account enumeration)।

</details>

**৪. `user_id` কেন client থেকে নিই না?**

<details><summary>উত্তর</summary>

Client যে-কোনো সংখ্যা পাঠাতে পারে, অন্যের নামে কাজ করতে পারত। Token-এর পরিচয় `SECRET_KEY` দিয়ে সই করা, নকল করা যায় না। তাই সবসময় `current_user.id`।

</details>

**৫. অন্যের conversation চাইলে 403 না দিয়ে 404 কেন?**

<details><summary>উত্তর</summary>

403 বলে দেয় ওই id-র conversation আছে। 404-এ "নেই" আর "অন্যের" একই রকম দেখায়, কোনো তথ্য ফাঁস হয় না।

</details>

**৬. `/me`-তে `response_model` না দিলে কী ফেরত আসত?**

<details><summary>উত্তর</summary>

User-এর সব column, `hashed_password` সহ। কোনো error আসত না।

</details>

**৭. `model_config = {"from_attributes": True}` কেন লাগে?**

<details><summary>উত্তর</summary>

Endpoint SQLAlchemy object return করে। Pydantic সাধারণত dict পড়ে, এটা দিলে object-এর attribute (`user.id`) থেকেও পড়তে পারে।

</details>

**৮. JWT-তে `exp` কেন রাখি? Payload-এ password রাখা যায় না কেন?**

<details><summary>উত্তর</summary>

`exp` না থাকলে চুরি যাওয়া token চিরকাল চলত। Payload শুধু base64, encrypt করা না, যে-কেউ পড়তে পারে। তাই গোপন কিছু রাখা যাবে না।

</details>

**৯. প্রতিটা message-এ LLM-কে পুরো history কেন পাঠাই? এর অসুবিধা কী?**

<details><summary>উত্তর</summary>

LLM কিছু মনে রাখে না (stateless), তাই আগের কথা বোঝাতে পুরো list পাঠাতে হয়। অসুবিধা: প্রতিবার token খরচ বাড়ে, আর লম্বা conversation-এ context window ছাড়িয়ে যাবে। তাই trimming লাগবে।

</details>

**১০. `llm.py` 429 বা 5xx-এ retry করে, 400 বা 401-এ করে না কেন? আর তিনবারই ব্যর্থ হলে user কী পায়?**

<details><summary>উত্তর</summary>

429/5xx সাময়িক, একটু পর ঠিক হতে পারে। 400/401 মানে body বা key ভুল, আবার চেষ্টা করলেও একই ফল। তিনবারই ব্যর্থ হলে `raise_for_status()` exception তোলে, user পায় 500 (এটাই বাকি কাজের friendly error)।

</details>

<a id="p8"></a>

## ৮. বাকি কাজ

<a id="s26"></a>

### Checklist

**শেষ হয়েছে ✅**

- [x] Docker-এ PostgreSQL: `chat-db`, port 5434
- [x] `models.py`: `users`, `conversations`, `messages` table
- [x] `create_table.py` দিয়ে table বানানো
- [x] `.env` + `.gitignore`: secret Git-এর বাইরে
- [x] `database.py`: engine, `SessionLocal`, `get_db` (yield/finally)
- [x] `security.py`: bcrypt hash, JWT (30 মিনিট মেয়াদ)
- [x] `schemas.py`: Create আর Response আলাদা, `from_attributes`
- [x] `POST /signup`, `POST /login`, `GET /me`
- [x] `get_current_user` guard
- [x] `POST /conversations`, `GET /conversations`
- [x] `POST /conversations/{id}/messages`: history সহ LLM-এর উত্তর, দুটোই DB-তে
- [x] `GET /conversations/{id}/messages`
- [x] `llm.py`: 429/5xx-এ তিনবার পর্যন্ত চেষ্টা
- [x] Ownership check (404), এক রকম login error, hash কখনো response-এ না

**বাকি ⏳ (এই ক্রমে করবো)**

- [ ] **Streaming replies (SSE):** Week 08-এর `stream_text` idea আবার ব্যবহার: `stream=True`, প্রতিটা টুকরো `yield`, `StreamingResponse` দিয়ে `text/event-stream`। পুরো উত্তর জমিয়ে শেষে DB-তে save। খেয়াল রাখতে হবে: browser-এর `EventSource` header পাঠাতে পারে না, তাই token-সহ stream-এর জন্য `fetch` দিয়ে stream পড়তে হবে।
- [ ] **Usage tracking:** API response-এর `usage` (prompt_tokens, completion_tokens) প্রতিটা উত্তরের সাথে DB-তে save। নতুন column লাগবে, আর `create_all` পুরনো table বদলায় না, তাই Alembic বা table নতুন করে বানানো।
- [ ] **Model config per conversation:** `Conversation`-এ model, system prompt, temperature রাখা, আর `get_reply(messages, model=...)`-এ পাঠানো।
- [ ] **Structured-output mode:** উত্তর নির্দিষ্ট JSON আকারে চাওয়া (`response_format`), Pydantic দিয়ে যাচাই।
- [ ] **LLM ব্যর্থ হলে friendly error:** `get_reply`-কে `try/except httpx.HTTPError` দিয়ে ঘিরে 500-এর বদলে পরিষ্কার message (যেমন 503 "এখন উত্তর দেওয়া যাচ্ছে না, একটু পরে চেষ্টা করো")।
- [ ] **লম্বা history ছোট করা:** শেষের N টা message বা token গুনে context window-এর ভিতরে রাখা।
- [ ] **EmailStr validation:** `email: str` → `EmailStr` (`pydantic[email]` লাগবে)।
- [ ] **Auto conversation title:** প্রথম message থেকে নিজে title বানানো।
- [ ] **Delete conversation:** `DELETE /conversations/{id}`, ownership check সহ। আগে ওই conversation-এর message মুছতে হবে (ForeignKey), বা cascade।
- [ ] **Tests:** pytest + `TestClient`। `get_reply` mock করা (আসল API call না), `app.dependency_overrides[get_db]` দিয়ে আলাদা test DB।
- [ ] **UI:** প্রথমে Week 08-এর `chat.html` আবার ব্যবহার, পরে Next.js।
- [ ] **App-এর জন্য Docker:** `Dockerfile`, আর `docker compose` দিয়ে app + DB একসাথে।
- [ ] **Deployment আর README:** কীভাবে চালাতে হয়, কোন env variable লাগে (মান ছাড়া), endpoint-এর তালিকা।

> **এক লাইনে:** Backend-এর মূল কাঠামো শেষ। এখন দ্রুততা (streaming), খরচের হিসাব, নিরাপদ error, test, আর UI।